In [ ]:

!pip install -q transformers datasets accelerate

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [ ]:
# --- Quick check before writing cell 1 ---
import os
from pathlib import Path

# 1. Is Drive mounted?
drive_root = Path("/content/drive/MyDrive")
print(f"Drive mounted    : {drive_root.exists()}")

# 2. Does the folder exist?
splits_dir = drive_root / "persian_nlp_pipline"
print(f"Splits folder    : {splits_dir.exists()}")

# 3. List files in that folder
if splits_dir.exists():
    for f in sorted(splits_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name:<30} {size_mb:>8.1f} MB")

# 4. GPU check
import torch
print()
print(f"torch version    : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory (GB)  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}")

# 5. transformers version
import transformers
print()
print(f"transformers ver : {transformers.__version__}")

Drive mounted    : False
Splits folder    : False

torch version    : 2.14.1+cpu
CUDA available   : False

transformers ver : 5.22.1


In [ ]:
from pathlib import Path
drive_root = Path("/content/drive/MyDrive")
splits_dir = drive_root / "persian_nlp_pipline"
print(f"Drive mounted    : {drive_root.exists()}")
print(f"Splits folder    : {splits_dir.exists()}")
if splits_dir.exists():
    for f in sorted(splits_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name:<30} {size_mb:>8.1f} MB")

Drive mounted    : True
Splits folder    : True
  test.csv                          118.0 MB
  train_supervised.csv              458.1 MB
  validation.csv                    117.4 MB


In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 1: Setup
# ============================================================
# Environment: Google Colab, NVIDIA T4 (14.6 GB), Drive mounted.
#
# Goal:
#   - mount Drive (if not already)
#   - verify required packages
#   - fail-fast if no GPU
#   - define paths, seeds, hyperparameters in one place
#
# No files are written here. Models are saved in cell 7.
# ============================================================

# --- 1. Mount Drive (idempotent: safe to re-run) -------------------------
from pathlib import Path
import sys, os, json, random, datetime

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.exists(), "Google Drive is not mounted."

# --- 2. Verify required packages ----------------------------------------
REQUIRED = {
    "torch": None,
    "transformers": None,
    "datasets": None,
    "accelerate": None,
    "sklearn": "sklearn",
    "polars": None,
    "numpy": None,
}
missing = []
for import_name, pip_name in REQUIRED.items():
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"[ok] {import_name:<14} {version}")
    except ImportError:
        print(f"[MISSING] {import_name}")
        missing.append(pip_name or import_name)

if missing:
    print(f"\nInstalling missing packages: {missing}")
    import subprocess
    subprocess.run(["pip", "install", "-q", *missing], check=True)
    print("Installed. Please re-run this cell to verify.")
    raise SystemExit("Re-run cell after installing packages.")

# --- 3. GPU check (fail-fast) -------------------------------------------
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. This notebook is not practical on CPU. "
    "In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU."
)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print()
print(f"GPU              : {gpu_name}")
print(f"GPU memory (GB)  : {gpu_mem_gb:.1f}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"PyTorch version  : {torch.__version__}")

# --- 4. Version compatibility check -------------------------------------
import transformers
print(f"transformers ver : {transformers.__version__}")

_tf_major = int(transformers.__version__.split(".")[0])
if _tf_major >= 5:
    print("  NOTE: transformers >= 5 detected. "
          "Some TrainingArguments defaults changed; "
          "we pass explicit values for everything we rely on.")

# --- 5. Paths -----------------------------------------------------------
SPLITS_DIR = DRIVE_ROOT / "persian_nlp_pipline"

TRAIN_CSV = SPLITS_DIR / "train_supervised.csv"
VAL_CSV   = SPLITS_DIR / "validation.csv"
TEST_CSV  = SPLITS_DIR / "test.csv"

for _p in (TRAIN_CSV, VAL_CSV, TEST_CSV):
    assert _p.exists(), f"Missing split file: {_p}"

# Output directories (created here, written by later cells)
OUTPUT_ROOT = DRIVE_ROOT / "persian_nlp_pipline" / "artifacts"
REC_OUTPUT_DIR  = OUTPUT_ROOT / "bert_recommendation"
BAND_OUTPUT_DIR = OUTPUT_ROOT / "bert_rating_band"
for _d in (REC_OUTPUT_DIR, BAND_OUTPUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

METRICS_JSON = OUTPUT_ROOT / "transformer_metrics.json"

# --- 6. Seeds (one source of truth, chains Part A/B/C) ------------------
SPLIT_SEED = 2026
SUBSET_SEED = SPLIT_SEED
TRAIN_SEED = SPLIT_SEED

random.seed(TRAIN_SEED)
import numpy as np
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)
torch.cuda.manual_seed_all(TRAIN_SEED)

# --- 7. Model & training hyperparameters --------------------------------
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"  # ParsBERT v2.0
MAX_LENGTH = 128
SUBSET_SIZE = 500_000

# Training (per model)
BATCH_SIZE = 16
GRAD_ACCUM = 2          # effective batch size = 32
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
FP16 = True
EARLY_STOP_PATIENCE = 1
EVAL_STEPS = 2000        # evaluate every N optimizer steps
SAVE_STEPS = 2000      # checkpoint every N optimizer steps

# Labels
REC_LABELS  = ["recommended", "not_recommended", "no_idea"]
BAND_LABELS = ["low", "medium", "high"]

# --- 8. Ready message ---------------------------------------------------
print()
print("=" * 60)
print("Notebook 04 — environment ready.")
print("=" * 60)
print(f"SPLITS_DIR       : {SPLITS_DIR}")
print(f"TRAIN_CSV        : {TRAIN_CSV.name}  "
      f"({TRAIN_CSV.stat().st_size / 1024**2:.1f} MB)")
print(f"VAL_CSV          : {VAL_CSV.name}  "
      f"({VAL_CSV.stat().st_size / 1024**2:.1f} MB)")
print(f"TEST_CSV         : {TEST_CSV.name}  "
      f"({TEST_CSV.stat().st_size / 1024**2:.1f} MB)")
print()
print(f"OUTPUT_ROOT      : {OUTPUT_ROOT}")
print(f"REC_OUTPUT_DIR   : {REC_OUTPUT_DIR.name}/")
print(f"BAND_OUTPUT_DIR  : {BAND_OUTPUT_DIR.name}/")
print(f"METRICS_JSON     : {METRICS_JSON.name}")
print()
print(f"MODEL_NAME       : {MODEL_NAME}")
print(f"MAX_LENGTH       : {MAX_LENGTH}")
print(f"SUBSET_SIZE      : {SUBSET_SIZE:,}")
print(f"BATCH_SIZE       : {BATCH_SIZE}  (x{GRAD_ACCUM} accum = {BATCH_SIZE * GRAD_ACCUM})")
print(f"EPOCHS           : {EPOCHS}")
print(f"LEARNING_RATE    : {LEARNING_RATE}")
print(f"FP16             : {FP16}")
print(f"EARLY_STOP       : patience={EARLY_STOP_PATIENCE}")
print()
print(f"SPLIT_SEED       : {SPLIT_SEED}")

[ok] torch          2.14.1+cu130
[ok] transformers   5.22.1
[ok] datasets       4.8.5
[ok] accelerate     1.15.0
[ok] sklearn        1.6.1
[ok] polars         1.35.2
[ok] numpy          2.4.6

GPU              : Tesla T4
GPU memory (GB)  : 14.6
CUDA version     : 13.0
PyTorch version  : 2.14.1+cu130
transformers ver : 5.22.1
  NOTE: transformers >= 5 detected. Some TrainingArguments defaults changed; we pass explicit values for everything we rely on.

Notebook 04 — environment ready.
SPLITS_DIR       : /content/drive/MyDrive/persian_nlp_pipline
TRAIN_CSV        : train_supervised.csv  (458.1 MB)
VAL_CSV          : validation.csv  (117.4 MB)
TEST_CSV         : test.csv  (118.0 MB)

OUTPUT_ROOT      : /content/drive/MyDrive/persian_nlp_pipline/artifacts
REC_OUTPUT_DIR   : bert_recommendation/
BAND_OUTPUT_DIR  : bert_rating_band/
METRICS_JSON     : transformer_metrics.json

MODEL_NAME       : HooshvareLab/bert-fa-base-uncased
MAX_LENGTH       : 128
SUBSET_SIZE      : 500,000
BATCH_SIZE   

In [ ]:
import json
from pathlib import Path

manifest_path = Path("/content/drive/MyDrive/persian_nlp_pipline/split_manifest.json")
assert manifest_path.exists(), "manifest not uploaded yet"

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

print("Seeds        :", manifest["seeds"])
print("Counts       :", manifest["counts"])
print("Band thresh  :", manifest["rating_band"]["thresholds"])
print("Ordinal enc  :", manifest["rating_band"]["ordinal_encoding"])
print("Masking      :", manifest["masking"]["per_class"])

Seeds        : {'SPLIT_SEED': 2026, 'MASK_SEED': 'digikala-semi-supervised-v1', 'MASK_RATE': 0.2}
Counts       : {'total_clean_rows': 6149834, 'products': 331512, 'train_products': 232057, 'val_products': 49727, 'test_products': 49728, 'train_supervised': 3578692, 'train_pseudo_unlabeled': 737731, 'validation': 913570, 'test': 919841, 'natural_unlabeled': 627763}
Band thresh  : {'low': '0 <= rate < 2.5', 'medium': '2.5 <= rate < 4.0', 'high': '4.0 <= rate <= 5.0', 'invalid': 'rate missing or outside [0, 5]'}
Ordinal enc  : {'low': 0, 'medium': 1, 'high': 2}
Masking      : {'no_idea': {'total': 415723, 'masked': 83144, 'mask_ratio_actual': 0.199999}, 'not_recommended': {'total': 351901, 'masked': 70380, 'mask_ratio_actual': 0.199999}, 'recommended': {'total': 2921036, 'masked': 584207, 'mask_ratio_actual': 0.2}}


In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 2: Load & Stratified Subset
# ============================================================
# Goal:
#   1. (optional) verify counts against split_manifest.json
#   2. load the three split CSVs
#   3. build a 500K stratified subset of train_supervised
#      by joint stratification on (recommendation_status, rating_band)
#   4. write /content/train_subset.parquet for fast reload
#
# Writes: /content/train_subset.parquet (temporary, not in Drive)
# ============================================================

import polars as pl
import math
import time
from pathlib import Path

# --- 1. (Optional) verify against manifest if it was uploaded -----------
manifest_path = SPLITS_DIR / "split_manifest.json"
if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    print("Manifest found — verifying row counts.")
    print(f"  expected train_supervised : {manifest['counts']['train_supervised']:,}")
    print(f"  expected validation       : {manifest['counts']['validation']:,}")
    print(f"  expected test             : {manifest['counts']['test']:,}")
    print()
else:
    manifest = None
    print("No split_manifest.json found — skipping count verification.")
    print()

# --- 2. Load the three split CSVs ---------------------------------------
# We only need the columns used for training and for stratification.
NEEDED_COLS = [
    "id",
    "body_normalized",
    "recommendation_status",
    "rating_band",
    "is_valid_rating_band",
]

print("Loading split CSVs with polars...")
t0 = time.time()
train_full = pl.read_csv(
    TRAIN_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
).select(NEEDED_COLS)

val_full = pl.read_csv(
    VAL_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
).select(NEEDED_COLS)

test_full = pl.read_csv(
    TEST_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
).select(NEEDED_COLS)

t_load = time.time() - t0
print(f"  loaded in {t_load:.1f}s")
print(f"  train : {train_full.height:,}")
print(f"  val   : {val_full.height:,}")
print(f"  test  : {test_full.height:,}")

# Verify if manifest was present
if manifest is not None:
    assert train_full.height == manifest["counts"]["train_supervised"], "train count mismatch"
    assert val_full.height   == manifest["counts"]["validation"],       "val count mismatch"
    assert test_full.height  == manifest["counts"]["test"],             "test count mismatch"
    print("  counts match manifest. OK.")
print()

# --- 3. Keep only rows usable for joint supervision ---------------------
# Rationale: this subset is shared by Task A and Task B.
# Rows must have both labels. That costs us the ~627K rows that only have
# a rating_band label but no recommendation_status. For a shared subset,
# that is the right trade-off.
print("Filtering to rows with both labels present...")
t0 = time.time()
train_usable = train_full.filter(
    pl.col("recommendation_status").is_not_null()
    & pl.col("rating_band").is_not_null()
    & (pl.col("rating_band") != "invalid")
    & pl.col("body_normalized").is_not_null()
)
t_filter = time.time() - t0
print(f"  filtered in {t_filter:.1f}s")
print(f"  usable train rows : {train_usable.height:,}  "
      f"({100 * train_usable.height / train_full.height:.2f}% of train)")
print()

# --- 4. Joint stratification on (recommendation_status, rating_band) ----
# For each (rec, band) group, sample n_group = floor(SUBSET_SIZE * n_group / N).
# This preserves the marginal distribution of BOTH targets as closely as
# possible given the 500K budget.
print(f"Building {SUBSET_SIZE:,}-row stratified subset (seed={SUBSET_SEED})...")
t0 = time.time()

# Add a joint group key and compute counts
train_usable = train_usable.with_columns(
    (pl.col("recommendation_status") + "|" + pl.col("rating_band")).alias("_group")
)

group_counts = (
    train_usable
    .group_by("_group")
    .agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.col("n").sum() * SUBSET_SIZE)
        .floor()
        .cast(pl.Int64)
        .alias("n_sample")
    )
    .sort("_group")
)
print("  Joint group sizes and sample targets:")
print(group_counts)
print()

# Sample per group
sampled_parts = []
for row in group_counts.iter_rows(named=True):
    grp = row["_group"]
    n_take = row["n_sample"]
    if n_take <= 0:
        continue
    df_grp = train_usable.filter(pl.col("_group") == grp)
    part = df_grp.sample(n=n_take, seed=SUBSET_SEED, with_replacement=False)
    sampled_parts.append(part)

train_subset = pl.concat(sampled_parts).drop("_group")

# Because of floor(), we may be slightly under SUBSET_SIZE.
# Top up from the largest group if needed.
shortfall = SUBSET_SIZE - train_subset.height
if shortfall > 0:
    print(f"  topping up {shortfall:,} rows from the largest group...")
    biggest_group = group_counts.sort("n", descending=True).row(0, named=True)["_group"]
    already = train_subset.height
    candidates = (
        train_usable
        .filter(pl.col("_group") == biggest_group)
        .join(train_subset.select("id"), on="id", how="anti")
    )
    top_up = candidates.sample(
        n=min(shortfall, candidates.height),
        seed=SUBSET_SEED + 1,
        with_replacement=False,
    )
    train_subset = pl.concat([train_subset, top_up.drop("_group")])
    print(f"  after top-up: {train_subset.height:,} rows")

t_sample = time.time() - t0
print(f"  sampling completed in {t_sample:.1f}s")
print(f"  final subset size : {train_subset.height:,}")
print()

# --- 5. Report class distributions in the subset ----------------------
print("Subset class distributions:")
print("  recommendation_status:")
rec_dist_subset = (
    train_subset.group_by("recommendation_status")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / train_subset.height * 100).round(2).alias("pct"))
    .sort("recommendation_status")
)
print(rec_dist_subset)

print("  rating_band:")
band_dist_subset = (
    train_subset.group_by("rating_band")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / train_subset.height * 100).round(2).alias("pct"))
    .sort("rating_band")
)
print(band_dist_subset)

# --- 6. Write the subset to /content for fast reload ------------------
SUBSET_PARQUET = DRIVE_ROOT / "persian_nlp_pipline" / "artifacts" / "train_subset.parquet"
train_subset.write_parquet(SUBSET_PARQUET, compression="zstd")
print()
print(f"Wrote subset parquet : {SUBSET_PARQUET}")
print(f"  size               : {SUBSET_PARQUET.stat().st_size / 1024**2:.1f} MB")
print(f"  rows               : {train_subset.height:,}")
print()

# --- 7. Confirm val/test stay full (no subsetting) -------------------
assert val_full.height == manifest["counts"]["validation"] if manifest else True
print("Validation and test remain full (no subsetting).")
print()
print("=" * 60)
print("Subset ready. Next: tokenizer + Dataset (cell 3).")
print("=" * 60)

Manifest found — verifying row counts.
  expected train_supervised : 3,578,692
  expected validation       : 913,570
  expected test             : 919,841

Loading split CSVs with polars...
  loaded in 2.8s
  train : 3,578,692
  val   : 913,570
  test  : 919,841
  counts match manifest. OK.

Filtering to rows with both labels present...
  filtered in 0.1s
  usable train rows : 2,950,626  (82.45% of train)

Building 500,000-row stratified subset (seed=2026)...
  Joint group sizes and sample targets:
shape: (9, 3)
┌────────────────────────┬─────────┬──────────┐
│ _group                 ┆ n       ┆ n_sample │
│ ---                    ┆ ---     ┆ ---      │
│ str                    ┆ u32     ┆ i64      │
╞════════════════════════╪═════════╪══════════╡
│ no_idea|high           ┆ 57229   ┆ 9697     │
│ no_idea|low            ┆ 58230   ┆ 9867     │
│ no_idea|medium         ┆ 217102  ┆ 36789    │
│ not_recommended|high   ┆ 6980    ┆ 1182     │
│ not_recommended|low    ┆ 236694  ┆ 40109    │
│ 

In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 3: Tokenizer & Dataset
# ============================================================
# Goal:
#   - load the ParsBERT tokenizer once
#   - define label maps for both tasks (stable order)
#   - define a Dataset that tokenizes on-the-fly
#   - set up dynamic padding with DataCollatorWithPadding
#   - quick sanity check on 5 samples
#   - save the tokenizer to both output dirs in Drive
#
# Writes: REC_OUTPUT_DIR/tokenizer/  and  BAND_OUTPUT_DIR/tokenizer/
# ============================================================

import json
import time
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# --- 1. Load tokenizer (one time) ----------------------------------------
print(f"Loading tokenizer: {MODEL_NAME}")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  loaded in {time.time() - t0:.1f}s")
print(f"  vocab_size : {tokenizer.vocab_size:,}")

# Quick Persian sanity check: a Persian word must map to more than one [UNK].
_test = tokenizer("سلام دنیا این یک تست است", add_special_tokens=False)["input_ids"]
_n_unk = sum(1 for x in _test if x == tokenizer.unk_token_id)
print(f"  sample ids : {_test[:10]}...")
print(f"  unknown tokens in sample: {_n_unk}")
assert _n_unk <= 2, "tokenizer seems to mishandle Persian text"
print()

# --- 2. Label maps (stable order across training, eval, inference) -------
REC_LABEL2ID = {label: i for i, label in enumerate(REC_LABELS)}
REC_ID2LABEL = {i: label for label, i in REC_LABEL2ID.items()}

BAND_LABEL2ID = {label: i for i, label in enumerate(BAND_LABELS)}
BAND_ID2LABEL = {i: label for label, i in BAND_LABEL2ID.items()}

print(f"recommendation label map: {REC_LABEL2ID}")
print(f"rating_band label map   : {BAND_LABEL2ID}")
print()

# --- 3. Dataset class ----------------------------------------------------
class ReviewDataset(Dataset):
    """
    On-the-fly tokenization of Persian review bodies.

    Parameters
    ----------
    df          : polars.DataFrame with columns body_normalized,
                  recommendation_status, rating_band
    tokenizer   : a HuggingFace tokenizer
    max_length  : int
    rec2id      : dict[str, int]
    band2id     : dict[str, int]

    Returns
    -------
    dict with:
        input_ids       : list[int]
        attention_mask  : list[int]
        rec_label       : int
        band_label      : int
    """

    def __init__(self, df, tokenizer, max_length, rec2id, band2id):
        self.texts = df["body_normalized"].fill_null("").to_list()
        self.rec_labels = df["recommendation_status"].to_list()
        self.band_labels = df["rating_band"].to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.rec2id = rec2id
        self.band2id = band2id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,        # padding is done in the collator
            return_tensors=None,  # return Python lists
        )
        enc["rec_label"] = self.rec2id[self.rec_labels[idx]]
        enc["band_label"] = self.band2id[self.band_labels[idx]]
        return enc


# --- 4. Collator: dynamic padding + wrap labels for one task --------------
# We wrap DataCollatorWithPadding to accept our extra label keys
# (rec_label, band_label) and produce a HuggingFace-style batch with
# a single 'labels' field for the task at hand.
def make_collator(tokenizer, task: str):
    """
    task : "recommendation" or "rating_band"
    """
    assert task in ("recommendation", "rating_band")
    base = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    label_key = "rec_label" if task == "recommendation" else "band_label"

    def collate(batch):
        # Pull out the chosen task label before passing to the base collator.
        labels = torch.tensor([b.pop(label_key) for b in batch], dtype=torch.long)
        features = base(batch)                # input_ids, attention_mask
        features["labels"] = labels
        return features

    return collate


# --- 5. Quick sanity check on 5 samples ---------------------------------
print("Sanity check — tokenizing 5 samples from the subset:")
small = train_subset.head(5)
for i in range(small.height):
    text = small["body_normalized"][i]
    rec_lbl = small["recommendation_status"][i]
    band_lbl = small["rating_band"][i]
    enc = tokenizer(
        text, truncation=True, max_length=MAX_LENGTH,
        padding=False, return_tensors=None,
    )
    n_tok = len(enc["input_ids"])
    trunc = " [TRUNCATED]" if n_tok >= MAX_LENGTH else ""
    short = text[:60] + ("..." if len(text) > 60 else "")
    print(f"  text    : {short!r}")
    print(f"  tokens  : {n_tok}{trunc}")
    print(f"  labels  : rec={rec_lbl!r}  band={band_lbl!r}")
    print()

# --- 6. Token-length distribution on 5,000 sampled rows -----------------
print("Token-length distribution on 5,000 sampled rows...")
sample_for_lengths = train_subset.sample(n=5_000, seed=SUBSET_SEED)
lengths = []
t0 = time.time()
for text in sample_for_lengths["body_normalized"].to_list():
    enc = tokenizer(
        text if text is not None else "",
        truncation=True, max_length=MAX_LENGTH,
        padding=False, return_tensors=None,
    )
    lengths.append(len(enc["input_ids"]))
lengths = np.array(lengths)
t_tok = time.time() - t0

print(f"  tokenized 5,000 rows in {t_tok:.1f}s  "
      f"({1000 * t_tok / 5000:.1f} ms/row)")
print(f"  min  : {lengths.min()}")
print(f"  p50  : {int(np.percentile(lengths, 50))}")
print(f"  p90  : {int(np.percentile(lengths, 90))}")
print(f"  p95  : {int(np.percentile(lengths, 95))}")
print(f"  p99  : {int(np.percentile(lengths, 99))}")
print(f"  max  : {lengths.max()}")
print(f"  % at MAX_LENGTH (truncated): "
      f"{100 * (lengths >= MAX_LENGTH).sum() / len(lengths):.2f}%")
print()

# --- 7. Save the tokenizer to both output dirs in Drive -----------------
tok_rec_path  = REC_OUTPUT_DIR / "tokenizer"
tok_band_path = BAND_OUTPUT_DIR / "tokenizer"
tok_rec_path.mkdir(parents=True, exist_ok=True)
tok_band_path.mkdir(parents=True, exist_ok=True)

tokenizer.save_pretrained(tok_rec_path)
tokenizer.save_pretrained(tok_band_path)

print(f"Saved tokenizer to:")
print(f"  {tok_rec_path}")
print(f"  {tok_band_path}")
print()

# --- 8. Ready message ---------------------------------------------------
print("=" * 60)
print("Cell 3 complete: tokenizer + dataset + collator ready.")
print("Next: cell 4 (Task A — BERT fine-tuning for recommendation).")
print("=" * 60)

Loading tokenizer: HooshvareLab/bert-fa-base-uncased
  loaded in 0.8s
  vocab_size : 100,000
  sample ids : [4285, 3810, 2802, 2829, 7623, 2806]...
  unknown tokens in sample: 0

recommendation label map: {'recommended': 0, 'not_recommended': 1, 'no_idea': 2}
rating_band label map   : {'low': 0, 'medium': 1, 'high': 2}

Sanity check — tokenizing 5 samples from the subset:
  text    : 'از قالب بزرگ استفاده\u200cشده و مقداری گشاد بود به خصوص قسمت جلوی...'
  tokens  : 15
  labels  : rec='no_idea'  band='high'

  text    : '3 سال پیش برای صندلی اداره گرفتم. تا حالا که خوب بوده فقط مح...'
  tokens  : 25
  labels  : rec='no_idea'  band='high'

  text    : 'به نظر خوبه'
  tokens  : 5
  labels  : rec='no_idea'  band='high'

  text    : 'در توضیحات رنگ دلخواه را بنویسید'
  tokens  : 8
  labels  : rec='no_idea'  band='high'

  text    : 'خوب هست. ولی به پوسته طرف مقابل هم\u200cبستگی داره.'
  tokens  : 13
  labels  : rec='no_idea'  band='high'

Token-length distribution on 5,000 sampled rows...
 

In [ ]:
from pathlib import Path
contents = list(REC_OUTPUT_DIR.iterdir()) if REC_OUTPUT_DIR.exists() else []
print("REC_OUTPUT_DIR contents:", [p.name for p in contents])

REC_OUTPUT_DIR contents: ['tokenizer']


In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 4: Task A Training
# ============================================================
# transformers 5.22 compatible: no overwrite_output_dir, no warmup_ratio,
# no logging_dir, no disable_tqdm. Uses warmup_steps and a checkpoint
# helper that handles the "no checkpoint yet" case gracefully.
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model ------------------------------------------------------
print(f"Loading model: {MODEL_NAME}")
model_rec = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(REC_LABELS),
    id2label=REC_ID2LABEL,
    label2id=REC_LABEL2ID,
)
print(f"  total params: {sum(p.numel() for p in model_rec.parameters()):,}")

# --- 2. Datasets --------------------------------------------------------
train_ds_rec = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_a = val_full.filter(pl.col("recommendation_status").is_not_null())
val_ds_rec = ReviewDataset(val_full_task_a, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_rec = make_collator(tokenizer, task="recommendation")
print(f"  train: {len(train_ds_rec):,}  val: {len(val_ds_rec):,}")

# --- 3. Metrics ---------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 4. TrainingArguments (v5 compatible) ------------------------------
total_optimizer_steps = (len(train_ds_rec) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
print(f"  total steps: {total_optimizer_steps:,}  warmup: {warmup_steps:,}")

training_args_rec = TrainingArguments(
    output_dir=str(REC_OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)

# --- 5. Trainer --------------------------------------------------------
trainer_rec = Trainer(
    model=model_rec,
    args=training_args_rec,
    train_dataset=train_ds_rec,
    eval_dataset=val_ds_rec,
    data_collator=collate_rec,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

# --- 6. Train (resume if checkpoint exists, else start fresh) ----------
def latest_checkpoint(output_dir: Path) -> str | None:
    if not output_dir.exists():
        return None
    ckpts = [p for p in output_dir.iterdir()
             if p.is_dir() and p.name.startswith("checkpoint-")]
    if not ckpts:
        return None
    ckpts.sort(key=lambda p: int(p.name.split("-")[1]))
    return str(ckpts[-1])

ckpt = latest_checkpoint(REC_OUTPUT_DIR)
if ckpt is None:
    print("No checkpoint found — starting fresh.")
else:
    print(f"Resuming from checkpoint: {ckpt}")

print("Starting training...")
t0 = time.time()
train_result = trainer_rec.train(resume_from_checkpoint=ckpt)
print(f"Finished in {(time.time() - t0)/60:.1f} min")
print(f"  global_step: {train_result.global_step}")
print(f"  train_loss : {train_result.training_loss:.4f}")
if trainer_rec.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")
if trainer_rec.state.best_metric is not None:
    print(f"  best metric    : {trainer_rec.state.best_metric:.4f}")

Loading model: HooshvareLab/bert-fa-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

  total params: 162,843,651
  train: 500,000  val: 780,620
  total steps: 31,250  warmup: 3,125
No checkpoint found — starting fresh.
Starting training...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,0.756352,0.362696,0.858666,0.654328
4000,0.712524,0.353614,0.863110,0.676738
6000,0.752653,0.355130,0.862801,0.670378


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Finished in 55.1 min
  global_step: 6000
  train_loss : 0.7807
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-4000
  best metric    : 0.6767


In [ ]:
# Check if trainer_rec exists and its state
try:
    print(f"trainer_rec defined: True")
    print(f"  global_step : {trainer_rec.state.global_step}")
    print(f"  log_history : {len(trainer_rec.state.log_history)} entries")
    print(f"  train_dataset size : {len(trainer_rec.train_dataset):,}")
    print(f"  args.max_steps : {trainer_rec.args.max_steps}")
    print(f"  args.num_train_epochs : {trainer_rec.args.num_train_epochs}")
    print(f"  args.fp16 : {trainer_rec.args.fp16}")
    print(f"  args.eval_strategy : {trainer_rec.args.eval_strategy}")
except NameError:
    print("trainer_rec is NOT defined — error occurred before Trainer.__init__")

trainer_rec defined: True
trainer_rec is NOT defined — error occurred before Trainer.__init__


In [ ]:
from pathlib import Path
print("REC_OUTPUT_DIR contents:")
for p in sorted(REC_OUTPUT_DIR.iterdir()):
    if p.is_dir():
        size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
        print(f"  {p.name}/  ({size:.2f} GB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

REC_OUTPUT_DIR contents:
  checkpoint-4000/  (1.82 GB)
  checkpoint-6000/  (1.82 GB)
  tokenizer/  (0.00 GB)


In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 4 v3 (FINAL)
# ============================================================
# Resume with WeightedTrainer + LR separate, from a fine-tuned
# model (checkpoint-6000) but with a FRESH optimizer.
# ============================================================

import time
import os
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model from checkpoint-6000 --------------------------------
ckpt_6000 = REC_OUTPUT_DIR / "checkpoint-6000"
assert ckpt_6000.exists(), f"checkpoint-6000 not found at {ckpt_6000}"

# Remove optimizer/scheduler state (mismatch with our 2-group optimizer)
for fname in ["optimizer.pt", "scheduler.pt"]:
    fpath = ckpt_6000 / fname
    if fpath.exists():
        fpath.unlink()
        print(f"Removed: {fpath.name}")

print(f"Loading model from checkpoint-6000...")
model_rec = AutoModelForSequenceClassification.from_pretrained(
    str(ckpt_6000),
    num_labels=len(REC_LABELS),
    id2label=REC_ID2LABEL,
    label2id=REC_LABEL2ID,
)
print(f"  total params: {sum(p.numel() for p in model_rec.parameters()):,}")

# --- 2. Datasets ------------------------------------------------------
train_ds_rec = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_a = val_full.filter(pl.col("recommendation_status").is_not_null())
val_ds_rec = ReviewDataset(val_full_task_a, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_rec = make_collator(tokenizer, task="recommendation")
print(f"  train: {len(train_ds_rec):,}  val: {len(val_ds_rec):,}")

# --- 3. Class weights --------------------------------------------------
counts = train_subset["recommendation_status"].value_counts()
count_map = {row["recommendation_status"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(REC_LABELS)
class_weights = torch.tensor(
    [N / (n_classes * count_map[label]) for label in REC_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

print("\nClass weights:")
for label, w in zip(REC_LABELS, class_weights.tolist()):
    print(f"  {label:<18}  count={count_map[label]:>7,}  weight={w:.3f}")

# --- 4. WeightedTrainer ------------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with 2 param groups ---------------------------------
HEAD_LR = 1e-4
head_params = list(model_rec.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_rec.parameters() if id(p) not in head_param_ids]

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": head_params,    "lr": HEAD_LR,      "weight_decay": WEIGHT_DECAY},
])
print(f"\nOptimizer: AdamW with 2 param groups")
print(f"  encoder : {len(encoder_params)} tensors, lr={LEARNING_RATE}")
print(f"  head    : {len(head_params)} tensors, lr={HEAD_LR}")

# --- 6. Metrics -------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 7. TrainingArguments with max_steps (fresh optimizer) ------------
# max_steps=25,000 = roughly what remains to complete 2 epochs
# (31,250 total - 6,000 already done ≈ 25,250). We use 25,000.
training_args_rec = TrainingArguments(
    output_dir=str(REC_OUTPUT_DIR),
    max_steps=25_000,                       # ← instead of num_train_epochs
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=2_500,                     # 10% of 25,000
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)

# --- 8. Trainer -------------------------------------------------------
trainer_rec = WeightedTrainer(
    model=model_rec,
    args=training_args_rec,
    train_dataset=train_ds_rec,
    eval_dataset=val_ds_rec,
    data_collator=collate_rec,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# --- 9. Train from fresh optimizer (no resume_from_checkpoint) --------
# The model is already fine-tuned (loaded from checkpoint-6000),
# but the optimizer/scheduler start fresh. No global_step mismatch.
print(f"\nStarting training with fresh optimizer (max_steps=25,000)...")
t0 = time.time()
train_result = trainer_rec.train()       # ← NO resume_from_checkpoint
t_train = time.time() - t0

print(f"\nFinished in {t_train / 60:.1f} min")
print(f"  global_step: {train_result.global_step}")
print(f"  train_loss : {train_result.training_loss:.4f}")
if trainer_rec.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")
if trainer_rec.state.best_metric is not None:
    print(f"  best metric    : {trainer_rec.state.best_metric:.4f}")

# --- 10. List checkpoints --------------------------------------------
ckpts = sorted(
    [p for p in REC_OUTPUT_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in REC_OUTPUT_DIR ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

Removed: optimizer.pt
Removed: scheduler.pt
Loading model from checkpoint-6000...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  total params: 162,843,651
  train: 500,000  val: 780,620

Class weights:
  recommended         count=395,948  weight=0.421
  not_recommended     count= 47,699  weight=3.494
  no_idea             count= 56,353  weight=2.958

Optimizer: AdamW with 2 param groups
  encoder : 199 tensors, lr=2e-05
  head    : 2 tensors, lr=0.0001

Starting training with fresh optimizer (max_steps=25,000)...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,1.155707,0.620620,0.832515,0.693445
4000,0.960126,0.695440,0.836748,0.684249


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,1.155707,0.620620,0.832515,0.693445
4000,0.960126,0.695440,0.836748,0.684249
6000,1.035719,0.802278,0.847447,0.692413


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished in 59.5 min
  global_step: 6000
  train_loss : 1.0501
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-2000
  best metric    : 0.6934

Checkpoints in REC_OUTPUT_DIR (3):
  checkpoint-2000  (1.82 GB)
  checkpoint-4000  (1.82 GB)
  checkpoint-6000  (1.82 GB)


In [ ]:
import os
print(f"REC_OUTPUT_DIR: {REC_OUTPUT_DIR}")
print(f"Contents:")
for p in sorted(REC_OUTPUT_DIR.iterdir()):
    if p.is_dir():
        size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
        print(f"  {p.name}/  ({size_gb:.2f} GB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

REC_OUTPUT_DIR: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation
Contents:
  checkpoint-2000/  (1.82 GB)
  checkpoint-4000/  (1.82 GB)
  checkpoint-6000/  (1.82 GB)
  tokenizer/  (0.00 GB)


In [ ]:
print("\nFinal Task A results (v3, with class_weights):")
print(f"  best F1 Macro: {trainer_rec.state.best_metric:.4f}")
print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")


Final Task A results (v3, with class_weights):
  best F1 Macro: 0.6934
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-2000


In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 4 v5 (FULL 1 EPOCH)
# ============================================================
# Complete one full epoch (15,625 steps) from checkpoint-2000 (best F1).
# - learning_rate=1e-5 (lower, to reduce overfitting)
# - weight_decay=0.05 (higher, more regularization)
# - NO early stopping (to complete the epoch)
# - max_steps=15,625 (one full epoch)
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load from best checkpoint -------------------------------------
best_ckpt = REC_OUTPUT_DIR / "checkpoint-2000"
assert best_ckpt.exists(), f"checkpoint-2000 not found: {best_ckpt}"

# Remove optimizer/scheduler (mismatch with our optimizer)
for fname in ["optimizer.pt", "scheduler.pt"]:
    fpath = best_ckpt / fname
    if fpath.exists():
        fpath.unlink()
        print(f"Removed: {fpath.name}")

print(f"Loading from {best_ckpt.name}...")
model_rec = AutoModelForSequenceClassification.from_pretrained(
    str(best_ckpt),
    num_labels=len(REC_LABELS),
    id2label=REC_ID2LABEL,
    label2id=REC_LABEL2ID,
)
print(f"  total params: {sum(p.numel() for p in model_rec.parameters()):,}")

# --- 2. Datasets ------------------------------------------------------
train_ds_rec = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_a = val_full.filter(pl.col("recommendation_status").is_not_null())
val_ds_rec = ReviewDataset(val_full_task_a, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_rec = make_collator(tokenizer, task="recommendation")
print(f"  train: {len(train_ds_rec):,}  val: {len(val_ds_rec):,}")

# --- 3. Class weights -------------------------------------------------
counts = train_subset["recommendation_status"].value_counts()
count_map = {row["recommendation_status"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(REC_LABELS)
class_weights = torch.tensor(
    [N / (n_classes * count_map[label]) for label in REC_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

# --- 4. WeightedTrainer ----------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with lower LR ---------------------------------------
HEAD_LR = 5e-5   # from 1e-4
ENCODER_LR = 1e-5   # from 2e-5

head_params = list(model_rec.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_rec.parameters() if id(p) not in head_param_ids]

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": ENCODER_LR, "weight_decay": 0.05},
    {"params": head_params,    "lr": HEAD_LR,    "weight_decay": 0.05},
])
print(f"\nOptimizer: AdamW")
print(f"  encoder : lr={ENCODER_LR}, weight_decay=0.05")
print(f"  head    : lr={HEAD_LR}, weight_decay=0.05")

# --- 6. Metrics -------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 7. TrainingArguments (full 1 epoch, no early stopping) -----------
OUTPUT_DIR_V5 = REC_OUTPUT_DIR / "v5_full_epoch"
OUTPUT_DIR_V5.mkdir(parents=True, exist_ok=True)

training_args_rec = TrainingArguments(
    output_dir=str(OUTPUT_DIR_V5),
    max_steps=15_625,                   # ← یک epoch کامل
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=ENCODER_LR,
    weight_decay=0.05,
    warmup_steps=1_562,                 # 10% of 15,625
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)

# --- 8. Trainer (NO early stopping — we want a full epoch) -----------
trainer_rec = WeightedTrainer(
    model=model_rec,
    args=training_args_rec,
    train_dataset=train_ds_rec,
    eval_dataset=val_ds_rec,
    data_collator=collate_rec,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer, None),
    # NO EarlyStoppingCallback — we want the full 15,625 steps
)

# --- 9. Train ---------------------------------------------------------
print(f"\nTraining for one full epoch (15,625 steps)...")
t0 = time.time()
train_result = trainer_rec.train()
t_train = time.time() - t0

print(f"\nFinished in {t_train / 60:.1f} min")
print(f"  global_step: {train_result.global_step}")
print(f"  train_loss : {train_result.training_loss:.4f}")
if trainer_rec.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")
if trainer_rec.state.best_metric is not None:
    print(f"  best metric    : {trainer_rec.state.best_metric:.4f}")

# --- 10. List checkpoints --------------------------------------------
ckpts = sorted(
    [p for p in OUTPUT_DIR_V5.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in v5_full_epoch ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

Removed: optimizer.pt
Removed: scheduler.pt
Loading from checkpoint-2000...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  total params: 162,843,651
  train: 500,000  val: 780,620

Optimizer: AdamW
  encoder : lr=1e-05, weight_decay=0.05
  head    : lr=5e-05, weight_decay=0.05

Training for one full epoch (15,625 steps)...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,0.619517,0.937088,0.833716,0.683089
4000,0.947418,0.670847,0.837306,0.694896
6000,0.907503,0.799810,0.836003,0.691386
8000,1.142401,0.595697,0.801007,0.683265
10000,1.191680,0.595963,0.835289,0.701713
12000,1.214194,0.582351,0.823804,0.698499
14000,1.198849,0.579589,0.832706,0.701040
15625,1.168670,0.581749,0.826356,0.699014


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished in 159.2 min
  global_step: 15625
  train_loss : 1.0763
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/v5_full_epoch/checkpoint-10000
  best metric    : 0.7017

Checkpoints in v5_full_epoch (3):
  checkpoint-10000  (1.82 GB)
  checkpoint-14000  (1.82 GB)
  checkpoint-15625  (1.82 GB)


In [ ]:
# Check Colab GPU quota and session time
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Check how long this session has been running
import time
from datetime import datetime
import psutil
boot = datetime.fromtimestamp(psutil.boot_time())
uptime = datetime.now() - boot
print(f"\nSession uptime: {uptime}")

Wed Nov 11 17:47:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             33W /   70W |    4565MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Task B Training

In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 5: Task B Training
# ============================================================
# Fine-tune ParsBERT for rating_band (low/medium/high).
#
# Configuration (adapted from Task A lessons):
#   - start from MODEL_NAME (fresh, not from Task A checkpoint)
#   - encoder lr=2e-5, head lr=1e-4
#   - class_weights from training distribution (imbalance 3.8:1)
#   - fp16, batch 16 x grad_accum 2 = effective 32
#   - max_steps=15,625 (one epoch)
#   - EarlyStoppingCallback(patience=2)
#   - resume_from_checkpoint via helper (safe in Colab)
#
# Writes checkpoint-XXXX/ every 2000 steps into BAND_OUTPUT_DIR (Drive).
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model (fresh, not from Task A) ----------------------------
print(f"Loading model: {MODEL_NAME}")
t0 = time.time()
model_band = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(BAND_LABELS),
    id2label=BAND_ID2LABEL,
    label2id=BAND_LABEL2ID,
)
print(f"  loaded in {time.time() - t0:.1f}s")
total_params     = sum(p.numel() for p in model_band.parameters())
trainable_params = sum(p.numel() for p in model_band.parameters() if p.requires_grad)
print(f"  total params     : {total_params:,}")
print(f"  trainable params : {trainable_params:,}")
print()

# --- 2. Datasets ------------------------------------------------------
# train_subset already has both labels present.
# val_full: filter to valid rating_band (exclude 1 invalid row)
train_ds_band = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_b = val_full.filter(
    pl.col("is_valid_rating_band") & pl.col("rating_band").is_not_null()
)
val_ds_band = ReviewDataset(val_full_task_b, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_band = make_collator(tokenizer, task="rating_band")

print(f"  train rows (Task B) : {len(train_ds_band):,}")
print(f"  val rows   (Task B) : {len(val_ds_band):,}")
print()

# --- 3. Class weights (inverse frequency) -----------------------------
counts = train_subset["rating_band"].value_counts()
count_map = {row["rating_band"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(BAND_LABELS)
class_weights_band = torch.tensor(
    [N / (n_classes * count_map[label]) for label in BAND_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

print("Class weights (inverse frequency):")
for label, w in zip(BAND_LABELS, class_weights_band.tolist()):
    print(f"  {label:<8}  count={count_map[label]:>7,}  weight={w:.3f}")
print()

# --- 4. WeightedTrainer (same as Task A) ------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with separate LR for encoder and head ---------------
HEAD_LR = 1e-4
head_params = list(model_band.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_band.parameters() if id(p) not in head_param_ids]

optimizer_band = torch.optim.AdamW([
    {"params": encoder_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": head_params,    "lr": HEAD_LR,      "weight_decay": WEIGHT_DECAY},
])
print(f"Optimizer: AdamW")
print(f"  encoder : {len(encoder_params)} tensors, lr={LEARNING_RATE}")
print(f"  head    : {len(head_params)} tensors, lr={HEAD_LR}")
print()

# --- 6. Metrics (add ordinal distance error) --------------------------
# BAND_ORDINAL = {"low": 0, "medium": 1, "high": 2}
def compute_metrics_band(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    # Ordinal distance error: mean |true_ordinal - pred_ordinal|
    # labels here are integers (0=low, 1=medium, 2=high) from the model's label2id
    ord_err = float(np.mean(np.abs(labels - preds)))

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "ordinal_distance_error": ord_err,
    }

# --- 7. TrainingArguments (one epoch, v5-compatible) ------------------
BAND_TOTAL_STEPS = (len(train_ds_band) // (BATCH_SIZE * GRAD_ACCUM)) * 1   # 1 epoch
warmup_steps_band = max(1, int(BAND_TOTAL_STEPS * WARMUP_RATIO))

training_args_band = TrainingArguments(
    output_dir=str(BAND_OUTPUT_DIR),
    max_steps=BAND_TOTAL_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps_band,
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)
print(f"Training schedule:")
print(f"  max_steps      : {BAND_TOTAL_STEPS:,}  (1 epoch)")
print(f"  warmup_steps   : {warmup_steps_band:,}")
print(f"  eval every     : {EVAL_STEPS} steps")
print(f"  save every     : {SAVE_STEPS} steps")
print()

# --- 8. Trainer -------------------------------------------------------
trainer_band = WeightedTrainer(
    model=model_band,
    args=training_args_band,
    train_dataset=train_ds_band,
    eval_dataset=val_ds_band,
    data_collator=collate_band,
    compute_metrics=compute_metrics_band,
    class_weights=class_weights_band,
    optimizers=(optimizer_band, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# --- 9. Resume if checkpoint exists, else start fresh ----------------
def latest_checkpoint(output_dir: Path) -> str | None:
    if not output_dir.exists():
        return None
    ckpts = [p for p in output_dir.iterdir()
             if p.is_dir() and p.name.startswith("checkpoint-")]
    if not ckpts:
        return None
    ckpts.sort(key=lambda p: int(p.name.split("-")[1]))
    return str(ckpts[-1])

ckpt_band = latest_checkpoint(BAND_OUTPUT_DIR)
if ckpt_band is None:
    print("No checkpoint found — starting fresh.")
else:
    print(f"Resuming from checkpoint: {ckpt_band}")

print("\nStarting Task B training...")
t0 = time.time()
train_result_band = trainer_band.train(resume_from_checkpoint=ckpt_band)
t_train = time.time() - t0

print(f"\nFinished in {t_train / 60:.1f} min")
print(f"  global_step: {train_result_band.global_step}")
print(f"  train_loss : {train_result_band.training_loss:.4f}")
if trainer_band.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_band.state.best_model_checkpoint}")
if trainer_band.state.best_metric is not None:
    print(f"  best metric    : {trainer_band.state.best_metric:.4f}")

# --- 10. List checkpoints ---------------------------------------------
ckpts = sorted(
    [p for p in BAND_OUTPUT_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in BAND_OUTPUT_DIR ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

print("\n" + "=" * 60)
print("Task B training complete.")
print("Next: cell 6 (evaluation on test + confusion matrix).")
print("=" * 60)

Loading model: HooshvareLab/bert-fa-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

  loaded in 1.0s
  total params     : 162,843,651
  trainable params : 162,843,651

  train rows (Task B) : 500,000
  val rows   (Task B) : 913,570

Class weights (inverse frequency):
  low       count= 93,678  weight=1.779
  medium    count= 89,797  weight=1.856
  high      count=316,525  weight=0.527

Optimizer: AdamW
  encoder : 199 tensors, lr=2e-05
  head    : 2 tensors, lr=0.0001

Training schedule:
  max_steps      : 15,625  (1 epoch)
  warmup_steps   : 1,562
  eval every     : 2000 steps
  save every     : 2000 steps

No checkpoint found — starting fresh.

Starting Task B training...


Step,Training Loss,Validation Loss


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

artifacts = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts")
print(f"artifacts/ ({sum(f.stat().st_size for f in artifacts.rglob('*') if f.is_file()) / 1024**3:.2f} GB):\n")

for p in sorted(artifacts.iterdir()):
    if p.is_dir():
        size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
        print(f"  {p.name}/  ({size_gb:.2f} GB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

artifacts/ (8.51 GB):

  bert_rating_band/  (0.00 GB)
  bert_recommendation/  (8.50 GB)
  train_subset.parquet  (14.4 MB)


In [ ]:
rec = artifacts / "bert_recommendation"
if rec.exists():
    print(f"bert_recommendation/ ({sum(f.stat().st_size for f in rec.rglob('*') if f.is_file()) / 1024**3:.2f} GB):\n")
    for p in sorted(rec.iterdir()):
        if p.is_dir():
            size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
            print(f"  {p.name}/  ({size_gb:.2f} GB)")
        else:
            print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

bert_recommendation/ (8.50 GB):

  checkpoint-2000/  (0.61 GB)
  checkpoint-4000/  (1.82 GB)
  checkpoint-6000/  (1.82 GB)
  tokenizer/  (0.00 GB)
  v5_full_epoch/  (4.25 GB)


In [ ]:
v5 = rec / "v5_full_epoch"
if v5.exists():
    print(f"v5_full_epoch/ ({sum(f.stat().st_size for f in v5.rglob('*') if f.is_file()) / 1024**3:.2f} GB):\n")
    for p in sorted(v5.iterdir()):
        if p.is_dir():
            size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
            print(f"  {p.name}/  ({size_gb:.2f} GB)")
        else:
            print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

v5_full_epoch/ (4.25 GB):

  checkpoint-2000/  (1.82 GB)
  checkpoint-4000/  (1.82 GB)
  checkpoint-6000/  (0.61 GB)
  checkpoint-8000/  (0.00 GB)


In [ ]:
# Total Drive usage
import subprocess
result = subprocess.run(['du', '-sh', '/content/drive/MyDrive/*'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import subprocess
result = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', 'checkpoint-10000', '-type', 'd'],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "checkpoint-10000 not found anywhere")

checkpoint-10000 not found anywhere


In [ ]:
import shutil
from pathlib import Path

REC_DIR = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation")

# Create a dedicated directory for the best model (outside checkpoints)
best_dir = REC_DIR / "model_v5_best"
best_dir.mkdir(exist_ok=True, parents=True)

# Copy model weights + config from checkpoint-4000 v5
src = REC_DIR / "v5_full_epoch" / "checkpoint-4000"
for fname in ["model.safetensors", "config.json"]:
    src_file = src / fname
    dst_file = best_dir / fname
    if src_file.exists():
        shutil.copy(src_file, dst_file)
        print(f"Copied: {fname}  ({src_file.stat().st_size / 1024**2:.1f} MB)")
    else:
        print(f"Not found: {src_file}")

# Also copy the tokenizer (same tokenizer for all)
tokenizer_src = REC_DIR / "tokenizer"
if tokenizer_src.exists():
    shutil.copytree(tokenizer_src, best_dir / "tokenizer", dirs_exist_ok=True)
    print(f"Copied: tokenizer/")

# Report
print(f"\n{best_dir}:")
for p in sorted(best_dir.iterdir()):
    if p.is_file():
        print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")
    else:
        print(f"  {p.name}/")
print(f"\nTotal: {sum(f.stat().st_size for f in best_dir.rglob('*') if f.is_file()) / 1024**3:.2f} GB")

Copied: model.safetensors  (621.2 MB)
Copied: config.json  (0.0 MB)
Copied: tokenizer/

/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/model_v5_best:
  config.json  (0.0 MB)
  model.safetensors  (621.2 MB)
  tokenizer/

Total: 0.61 GB


In [ ]:
import shutil

to_delete = [
    REC_DIR / "checkpoint-2000",           # v1, ناقص
    REC_DIR / "checkpoint-4000",           # v1, F1=0.677
    REC_DIR / "checkpoint-6000",           # v1
    REC_DIR / "v5_full_epoch" / "checkpoint-2000",   # ناقص
    REC_DIR / "v5_full_epoch" / "checkpoint-6000",   # ناقص
    REC_DIR / "v5_full_epoch" / "checkpoint-8000",   # خالی
    # نگه دار: v5_full_epoch/checkpoint-4000 (بهترین checkpoint موجود)
]

for p in to_delete:
    if p.exists():
        shutil.rmtree(p)
        print(f"Deleted: {p}")
    else:
        print(f"Not found: {p}")

# Report total
print(f"\nTotal after cleanup:")
total = sum(f.stat().st_size for f in REC_DIR.rglob("*") if f.is_file()) / 1024**3
print(f"  bert_recommendation/: {total:.2f} GB")

Deleted: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-2000
Deleted: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-4000
Deleted: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-6000
Deleted: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/v5_full_epoch/checkpoint-2000
Deleted: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/v5_full_epoch/checkpoint-6000
Deleted: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/v5_full_epoch/checkpoint-8000

Total after cleanup:
  bert_recommendation/: 2.43 GB


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

src = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/v5_full_epoch/checkpoint-4000")
zip_path = Path("/content/model_taskA_best")

shutil.make_archive(str(zip_path), 'zip', str(src))
files.download(str(zip_path) + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Part C — BERT Task B (rating_band)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA: True
GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
from pathlib import Path
import polars as pl

target = Path("/content/drive/MyDrive/persian_nlp_pipline")
print(f"Target: {target}")
print(f"Exists: {target.exists()}\n")

if target.exists():
    print("Files in target:")
    for p in sorted(target.iterdir()):
        if p.is_dir():
            size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
            print(f"  [dir]  {p.name}/  ({size_gb:.2f} GB)")
        else:
            print(f"  [file] {p.name}  ({p.stat().st_size/1024**2:.1f} MB)")

print("\nTesting CSV readability:")
for fname in ["train_supervised.csv", "validation.csv", "test.csv"]:
    f = target / fname
    if not f.exists():
        print(f"  {fname}: MISSING")
        continue
    try:
        df = pl.read_csv(
            f, separator=",", has_header=True,
            null_values=["", "nan", "NaN", "null", "None"],
            encoding="utf8-lossy", n_rows=1000,
        )
        print(f"  {fname}: OK ({df.height} rows)")
    except Exception as e:
        print(f"  {fname}: FAILED — {e}")

Target: /content/drive/MyDrive/persian_nlp_pipline
Exists: True

Files in target:
  [file] split_manifest.json  (0.0 MB)
  [file] test.csv  (118.0 MB)
  [file] train_supervised.csv  (458.1 MB)
  [file] validation.csv  (117.4 MB)

Testing CSV readability:
  train_supervised.csv: OK (1000 rows)
  validation.csv: OK (1000 rows)
  test.csv: OK (1000 rows)


Setup

In [ ]:
# ============================================================
# Cell 1: Setup (mount, GPU check, paths, seeds)
# ============================================================

from pathlib import Path
import json, random
import numpy as np
import torch, polars as pl, transformers

# --- Paths ---
DRIVE_ROOT = Path("/content/drive/MyDrive")
SPLITS_DIR = DRIVE_ROOT / "persian_nlp_pipline"

TRAIN_CSV = SPLITS_DIR / "train_supervised.csv"
VAL_CSV   = SPLITS_DIR / "validation.csv"
TEST_CSV  = SPLITS_DIR / "test.csv"
MANIFEST_PATH = SPLITS_DIR / "split_manifest.json"

for _p in (TRAIN_CSV, VAL_CSV, TEST_CSV, MANIFEST_PATH):
    assert _p.exists(), f"Missing: {_p}"

OUTPUT_ROOT     = SPLITS_DIR / "artifacts"
REC_OUTPUT_DIR  = OUTPUT_ROOT / "bert_recommendation"   # ← این خط اضافه شد
BAND_OUTPUT_DIR = OUTPUT_ROOT / "bert_rating_band"
REC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BAND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_JSON = OUTPUT_ROOT / "transformer_metrics.json"

# --- Seeds ---
SPLIT_SEED  = 2026
SUBSET_SEED = SPLIT_SEED
TRAIN_SEED  = SPLIT_SEED
random.seed(TRAIN_SEED); np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED); torch.cuda.manual_seed_all(TRAIN_SEED)

# --- Hyperparameters ---
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
MAX_LENGTH = 128
SUBSET_SIZE = 500_000
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS = 1
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
FP16 = True
EARLY_STOP_PATIENCE = 2
EVAL_STEPS = 2000
SAVE_STEPS = 2000

REC_LABELS  = ["recommended", "not_recommended", "no_idea"]
BAND_LABELS = ["low", "medium", "high"]

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"transformers: {transformers.__version__}, polars: {pl.__version__}")
print(f"REC_OUTPUT_DIR  : {REC_OUTPUT_DIR}")
print(f"BAND_OUTPUT_DIR : {BAND_OUTPUT_DIR}")
print(f"MODEL_NAME      : {MODEL_NAME}")
print(f"BATCH_SIZE      : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")
print(f"EPOCHS          : {EPOCHS}")
print(f"LEARNING_RATE   : {LEARNING_RATE}")
print(f"SPLIT_SEED      : {SPLIT_SEED}")

GPU: Tesla T4
transformers: 5.22.1, polars: 1.35.2
REC_OUTPUT_DIR  : /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation
BAND_OUTPUT_DIR : /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_rating_band
MODEL_NAME      : HooshvareLab/bert-fa-base-uncased
BATCH_SIZE      : 16 x 2 = 32
EPOCHS          : 1
LEARNING_RATE   : 2e-05
SPLIT_SEED      : 2026


 Load Splits + Build 500K Subset

In [ ]:
# ============================================================
# Cell 2: Load splits + build the same 500K stratified subset
# ============================================================
# Reuses the exact same subset as Task A (deterministic, seed=2026).
# Uses encoding="utf8-lossy" to survive any corrupt bytes in the CSV.
# ============================================================

import polars as pl
import time
from pathlib import Path

# Load manifest
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

# Load splits (only needed columns)
NEEDED_COLS = ["id", "body_normalized", "recommendation_status",
               "rating_band", "is_valid_rating_band"]

print("Loading splits...")
t0 = time.time()
train_full = pl.read_csv(
    TRAIN_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
    encoding="utf8-lossy",
).select(NEEDED_COLS)
val_full = pl.read_csv(
    VAL_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
    encoding="utf8-lossy",
).select(NEEDED_COLS)
test_full = pl.read_csv(
    TEST_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
    encoding="utf8-lossy",
).select(NEEDED_COLS)
print(f"  loaded in {time.time()-t0:.1f}s")
print(f"  train={train_full.height:,} val={val_full.height:,} test={test_full.height:,}")

# Verify counts against manifest
assert train_full.height == manifest["counts"]["train_supervised"]
assert val_full.height == manifest["counts"]["validation"]
assert test_full.height == manifest["counts"]["test"]
print("  counts match manifest. OK.")

# Filter to usable rows (both labels present)
train_usable = train_full.filter(
    pl.col("recommendation_status").is_not_null()
    & pl.col("rating_band").is_not_null()
    & (pl.col("rating_band") != "invalid")
    & pl.col("body_normalized").is_not_null()
)
print(f"  usable train: {train_usable.height:,}")

# Build the same 500K stratified subset (joint on recommendation_status + rating_band)
SUBSET_SIZE = 500_000
SUBSET_SEED = 2026

train_usable = train_usable.with_columns(
    (pl.col("recommendation_status") + "|" + pl.col("rating_band")).alias("_group")
)
group_counts = (
    train_usable.group_by("_group").agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.col("n").sum() * SUBSET_SIZE)
        .floor().cast(pl.Int64).alias("n_sample")
    )
)
sampled_parts = []
for row in group_counts.iter_rows(named=True):
    if row["n_sample"] <= 0:
        continue
    part = train_usable.filter(pl.col("_group") == row["_group"]).sample(
        n=row["n_sample"], seed=SUBSET_SEED, with_replacement=False
    )
    sampled_parts.append(part)
train_subset = pl.concat(sampled_parts).drop("_group")

# Top up to exactly 500K
shortfall = SUBSET_SIZE - train_subset.height
if shortfall > 0:
    biggest = group_counts.sort("n", descending=True).row(0, named=True)["_group"]
    cand = train_usable.filter(pl.col("_group") == biggest).join(
        train_subset.select("id"), on="id", how="anti"
    )
    top_up = cand.sample(n=min(shortfall, cand.height), seed=SUBSET_SEED+1)
    train_subset = pl.concat([train_subset, top_up.drop("_group")])

print(f"  subset: {train_subset.height:,} rows")
print(f"  rating_band dist:")
print(train_subset.group_by("rating_band").agg(pl.len().alias("n")).sort("rating_band"))

# Save to Drive for reuse
SUBSET_PATH = OUTPUT_ROOT / "train_subset.parquet"
train_subset.write_parquet(SUBSET_PATH, compression="zstd")
print(f"  saved to {SUBSET_PATH} ({SUBSET_PATH.stat().st_size/1024**2:.1f} MB)")

Loading splits...
  loaded in 16.3s
  train=3,578,692 val=913,570 test=919,841
  counts match manifest. OK.
  usable train: 2,950,626
  subset: 500,000 rows
  rating_band dist:
shape: (3, 2)
┌─────────────┬────────┐
│ rating_band ┆ n      │
│ ---         ┆ ---    │
│ str         ┆ u32    │
╞═════════════╪════════╡
│ high        ┆ 316525 │
│ low         ┆ 93678  │
│ medium      ┆ 89797  │
└─────────────┴────────┘
  saved to /content/drive/MyDrive/persian_nlp_pipline/artifacts/train_subset.parquet (14.3 MB)


Tokenizer + Dataset + Collator

In [ ]:
# ============================================================
# Cell 3: Tokenizer + Dataset + Collator
# ============================================================

import time
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# --- 1. Load tokenizer (once) -------------------------------------------
print(f"Loading tokenizer: {MODEL_NAME}")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  loaded in {time.time()-t0:.1f}s  (vocab={tokenizer.vocab_size:,})")

# Quick sanity check: Persian must not map to [UNK]
_test = tokenizer("سلام دنیا این یک تست است", add_special_tokens=False)["input_ids"]
_n_unk = sum(1 for x in _test if x == tokenizer.unk_token_id)
assert _n_unk <= 2, f"tokenizer mishandles Persian ({_n_unk} UNKs)"
print(f"  Persian sanity check OK ({_n_unk} UNKs)")

# --- 2. Label maps ------------------------------------------------------
REC_LABEL2ID  = {l: i for i, l in enumerate(REC_LABELS)}
REC_ID2LABEL  = {i: l for l, i in REC_LABEL2ID.items()}
BAND_LABEL2ID = {l: i for i, l in enumerate(BAND_LABELS)}
BAND_ID2LABEL = {i: l for l, i in BAND_LABEL2ID.items()}
print(f"  rec  map: {REC_LABEL2ID}")
print(f"  band map: {BAND_LABEL2ID}")

# --- 3. Dataset ---------------------------------------------------------
class ReviewDataset(Dataset):
    """On-the-fly tokenization of Persian review bodies.

    Missing labels (None) are mapped to -100 (ignored by CrossEntropyLoss).
    This allows one Dataset to be used for both tasks, even when the
    "other" label is null on some rows.
    """
    IGNORE_INDEX = -100

    def __init__(self, df, tokenizer, max_length, rec2id, band2id):
        self.texts = df["body_normalized"].fill_null("").to_list()
        self.rec_labels = df["recommendation_status"].to_list()
        self.band_labels = df["rating_band"].to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.rec2id = rec2id
        self.band2id = band2id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        rec = self.rec_labels[idx]
        band = self.band_labels[idx]
        enc["rec_label"]  = self.rec2id[rec]  if rec  is not None else self.IGNORE_INDEX
        enc["band_label"] = self.band2id[band] if band is not None else self.IGNORE_INDEX
        return enc

# --- 4. Collator --------------------------------------------------------
def make_collator(tokenizer, task):
    assert task in ("recommendation", "rating_band")
    base = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    label_key = "rec_label" if task == "recommendation" else "band_label"

    def collate(batch):
        labels = torch.tensor([b.pop(label_key) for b in batch], dtype=torch.long)
        features = base(batch)
        features["labels"] = labels
        return features

    return collate

# --- 5. Save tokenizer to both output dirs ------------------------------
tok_rec_path  = REC_OUTPUT_DIR / "tokenizer"
tok_band_path = BAND_OUTPUT_DIR / "tokenizer"
tok_rec_path.mkdir(parents=True, exist_ok=True)
tok_band_path.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(tok_rec_path)
tokenizer.save_pretrained(tok_band_path)
print(f"  tokenizer saved to: {tok_rec_path.name}/ and {tok_band_path.name}/")

# --- 6. Quick smoke test on 5 samples -----------------------------------
print("\nSanity check (5 samples):")
for i in range(5):
    text = train_subset["body_normalized"][i]
    rec  = train_subset["recommendation_status"][i]
    band = train_subset["rating_band"][i]
    enc  = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding=False)
    n_tok = len(enc["input_ids"])
    trunc = " [TRUNC]" if n_tok >= MAX_LENGTH else ""
    print(f"  {n_tok:>3} tok{trunc:>7}  rec={rec!r:<18} band={band!r:<8}  {text[:50]!r}")

# --- 7. Token-length distribution on 5K rows ----------------------------
print("\nToken-length distribution (5,000 samples):")
sample = train_subset.sample(n=5_000, seed=SUBSET_SEED)
lengths = np.array([
    len(tokenizer(t if t is not None else "",
                  truncation=True, max_length=MAX_LENGTH,
                  padding=False)["input_ids"])
    for t in sample["body_normalized"].to_list()
])
print(f"  min={lengths.min()}  p50={int(np.percentile(lengths,50))}  "
      f"p90={int(np.percentile(lengths,90))}  p95={int(np.percentile(lengths,95))}  "
      f"p99={int(np.percentile(lengths,99))}  max={lengths.max()}")
print(f"  % at MAX_LENGTH: {100*(lengths >= MAX_LENGTH).sum()/len(lengths):.2f}%")

print("\n" + "=" * 60)
print("Cell 3 complete: tokenizer + dataset + collator ready.")
print("Next: cell 4 (Task B — BERT fine-tuning for rating_band).")
print("=" * 60)

Loading tokenizer: HooshvareLab/bert-fa-base-uncased
  loaded in 0.4s  (vocab=100,000)
  Persian sanity check OK (0 UNKs)
  rec  map: {'recommended': 0, 'not_recommended': 1, 'no_idea': 2}
  band map: {'low': 0, 'medium': 1, 'high': 2}
  tokenizer saved to: tokenizer/ and tokenizer/

Sanity check (5 samples):
    9 tok         rec='recommended'      band='high'    'مثل عکس بود و کیفیتش هم عالی'
    6 tok         rec='recommended'      band='high'    'کیفیتش مطلوبه.'
    9 tok         rec='recommended'      band='high'    'دفترخوبی بود خریدش رو توصیه می\u200cکنم'
    6 tok         rec='recommended'      band='high'    'فوق\u200cالعاده شیک و زیبا'
    5 tok         rec='recommended'      band='high'    'ویتالیر عالیه'

Token-length distribution (5,000 samples):
  min=3  p50=9  p90=25  p95=36  p99=64  max=128
  % at MAX_LENGTH: 0.08%

Cell 3 complete: tokenizer + dataset + collator ready.
Next: cell 4 (Task B — BERT fine-tuning for rating_band).


In [ ]:
# ============================================================
# Cell 4: Task B — Fine-tune ParsBERT for rating_band
# ============================================================
# Fresh model (not from Task A checkpoint).
# class_weights + separate LR for encoder/head.
# max_steps=15,625 (one epoch).
# save_total_limit=None -> keep ALL checkpoints (so best one never gets deleted).
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model (fresh) ---------------------------------------------
print(f"Loading fresh model: {MODEL_NAME}")
t0 = time.time()
model_band = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(BAND_LABELS),
    id2label=BAND_ID2LABEL,
    label2id=BAND_LABEL2ID,
)
print(f"  loaded in {time.time()-t0:.1f}s")
print(f"  total params: {sum(p.numel() for p in model_band.parameters()):,}")

# --- 2. Datasets -------------------------------------------------------
train_ds_band = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_b = val_full.filter(
    pl.col("is_valid_rating_band") & pl.col("rating_band").is_not_null()
)
val_ds_band = ReviewDataset(val_full_task_b, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_band = make_collator(tokenizer, task="rating_band")
print(f"  train: {len(train_ds_band):,}  val: {len(val_ds_band):,}")

# --- 3. Class weights (inverse frequency) -----------------------------
counts = train_subset["rating_band"].value_counts()
count_map = {row["rating_band"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(BAND_LABELS)
class_weights_band = torch.tensor(
    [N / (n_classes * count_map[l]) for l in BAND_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

print("\nClass weights:")
for label, w in zip(BAND_LABELS, class_weights_band.tolist()):
    print(f"  {label:<8}  count={count_map[label]:>7,}  weight={w:.3f}")

# --- 4. WeightedTrainer ------------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with separate LR -------------------------------------
HEAD_LR = 1e-4
head_params = list(model_band.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_band.parameters() if id(p) not in head_param_ids]

optimizer_band = torch.optim.AdamW([
    {"params": encoder_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": head_params,    "lr": HEAD_LR,      "weight_decay": WEIGHT_DECAY},
])
print(f"\nOptimizer: AdamW")
print(f"  encoder: {len(encoder_params)} tensors, lr={LEARNING_RATE}")
print(f"  head   : {len(head_params)} tensors, lr={HEAD_LR}")

# --- 6. Metrics (add ordinal distance error) ---------------------------
def compute_metrics_band(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "ordinal_distance_error": float(np.mean(np.abs(labels - preds))),
    }

# --- 7. TrainingArguments (1 epoch, keep ALL checkpoints) -------------
BAND_TOTAL_STEPS = (len(train_ds_band) // (BATCH_SIZE * GRAD_ACCUM)) * 1
warmup_steps_band = max(1, int(BAND_TOTAL_STEPS * WARMUP_RATIO))

training_args_band = TrainingArguments(
    output_dir=str(BAND_OUTPUT_DIR),
    max_steps=BAND_TOTAL_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps_band,
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=None,        # ← keep ALL checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)
print(f"\nSchedule: max_steps={BAND_TOTAL_STEPS:,}, warmup={warmup_steps_band:,}, "
      f"eval/save every {EVAL_STEPS} steps")

# --- 8. Trainer --------------------------------------------------------
trainer_band = WeightedTrainer(
    model=model_band,
    args=training_args_band,
    train_dataset=train_ds_band,
    eval_dataset=val_ds_band,
    data_collator=collate_band,
    compute_metrics=compute_metrics_band,
    class_weights=class_weights_band,
    optimizers=(optimizer_band, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

# --- 9. Resume if checkpoint exists, else fresh ------------------------
def latest_checkpoint(d):
    if not d.exists(): return None
    ckpts = [p for p in d.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")]
    if not ckpts: return None
    ckpts.sort(key=lambda p: int(p.name.split("-")[1]))
    return str(ckpts[-1])

ckpt = latest_checkpoint(BAND_OUTPUT_DIR)
print(f"Resume from: {ckpt or 'fresh start'}")

# --- 10. Train ---------------------------------------------------------
print("\nStarting Task B training...")
t0 = time.time()
result = trainer_band.train(resume_from_checkpoint=ckpt)
print(f"\nFinished in {(time.time()-t0)/60:.1f} min")
print(f"  global_step: {result.global_step}")
print(f"  train_loss : {result.training_loss:.4f}")
if trainer_band.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_band.state.best_model_checkpoint}")
if trainer_band.state.best_metric is not None:
    print(f"  best F1 Macro: {trainer_band.state.best_metric:.4f}")

# --- 11. List checkpoints ---------------------------------------------
ckpts = sorted(
    [p for p in BAND_OUTPUT_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in BAND_OUTPUT_DIR ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

print("\n" + "=" * 60)
print("Task B training complete.")
print("Next: cell 5 (evaluation on test + confusion matrix).")
print("=" * 60)

Loading fresh model: HooshvareLab/bert-fa-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

  loaded in 0.4s
  total params: 162,843,651
  train: 500,000  val: 913,570

Class weights:
  low       count= 93,678  weight=1.779
  medium    count= 89,797  weight=1.856
  high      count=316,525  weight=0.527

Optimizer: AdamW
  encoder: 199 tensors, lr=2e-05
  head   : 2 tensors, lr=0.0001

Schedule: max_steps=15,625, warmup=1,562, eval/save every 2000 steps
Resume from: fresh start

Starting Task B training...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Ordinal Distance Error
2000,1.854648,0.901985,0.596502,0.538103,0.491787
4000,1.782999,0.894262,0.693570,0.565526,0.430604
6000,1.782163,0.878630,0.659614,0.572327,0.443177
8000,1.770805,0.877522,0.650977,0.568722,0.458189
10000,1.810302,0.878087,0.651257,0.569919,0.462724


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished in 108.9 min
  global_step: 10000
  train_loss : 1.8205
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_rating_band/checkpoint-6000
  best F1 Macro: 0.5723

Checkpoints in BAND_OUTPUT_DIR (5):
  checkpoint-2000  (1.82 GB)
  checkpoint-4000  (1.82 GB)
  checkpoint-6000  (1.82 GB)
  checkpoint-8000  (1.82 GB)
  checkpoint-10000  (1.82 GB)

Task B training complete.
Next: cell 5 (evaluation on test + confusion matrix).


In [ ]:
import shutil
from pathlib import Path

best_ckpt = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_rating_band/checkpoint-6000")
best_dir = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_rating_band/model_band_best")
best_dir.mkdir(exist_ok=True)

for f in ["model.safetensors", "config.json"]:
    src = best_ckpt / f
    if src.exists():
        shutil.copy(src, best_dir / f)
        print(f"Copied: {f}  ({src.stat().st_size/1024**2:.1f} MB)")

# Copy tokenizer
tok_src = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_rating_band/tokenizer")
if tok_src.exists():
    shutil.copytree(tok_src, best_dir / "tokenizer", dirs_exist_ok=True)
    print("Copied: tokenizer/")

print(f"\nBest model saved to {best_dir}")

Copied: model.safetensors  (621.2 MB)
Copied: config.json  (0.0 MB)
Copied: tokenizer/

Best model saved to /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_rating_band/model_band_best


GPU 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA: True
GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/persian_nlp_pipline")
print(f"Base exists: {base.exists()}\n")

if base.exists():
    print("Contents of persian_nlp_pipline/:")
    for p in sorted(base.iterdir()):
        if p.is_dir():
            size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
            print(f"  [dir]  {p.name}/  ({size_gb:.2f} GB)")
        else:
            print(f"  [file] {p.name}  ({p.stat().st_size/1024**2:.1f} MB)")

# اگر پوشه پیدا نشد، جستجو کن
if not base.exists():
    import subprocess
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-maxdepth', '3', '-name', '*persian*', '-o', '-name', '*digikala*'],
        capture_output=True, text=True
    )
    print("Search results:")
    print(result.stdout if result.stdout else "nothing found")

Base exists: True

Contents of persian_nlp_pipline/:
  [dir]  artifacts/  (1.24 GB)
  [file] split_manifest.json  (0.0 MB)
  [file] test.csv  (118.0 MB)
  [file] train_pseudo_unlabeled.csv  (68.8 MB)
  [file] train_supervised.csv  (458.1 MB)
  [file] validation.csv  (117.4 MB)


In [ ]:
import polars as pl
from pathlib import Path

base = Path("/content/drive/MyDrive/persian_nlp_pipline")

# چک فایل‌های CSV
print("CSV files:")
for fname in ["train_supervised.csv", "train_pseudo_unlabeled.csv",
              "validation.csv", "test.csv"]:
    f = base / fname
    if not f.exists():
        print(f"  {fname}: ❌ MISSING")
        continue
    try:
        df = pl.read_csv(f, separator=",", has_header=True,
                         null_values=["", "nan", "NaN", "null", "None"],
                         encoding="utf8-lossy", n_rows=1000)
        print(f"  {fname}: ✅ OK ({df.height} rows read, {f.stat().st_size/1024**2:.1f} MB)")
    except Exception as e:
        print(f"  {fname}: ❌ FAILED — {e}")

# چک پوشه مدل Task A
print("\nTask A model:")
model_dir = base / "artifacts" / "bert_recommendation" / "model_v5_best"
if model_dir.exists():
    for f in sorted(model_dir.iterdir()):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size/1024**2:.1f} MB)")
        else:
            print(f"  {f.name}/")
else:
    print(f"  ❌ MISSING: {model_dir}")
    # جستجوی جایگزین
    import subprocess
    result = subprocess.run(
        ['find', str(base), '-name', 'model*safetensors', '-o', '-name', 'model_band_best'],
        capture_output=True, text=True
    )
    print(f"  Alternatives found:\n{result.stdout}")

CSV files:
  train_supervised.csv: ✅ OK (1000 rows read, 458.1 MB)
  train_pseudo_unlabeled.csv: ✅ OK (1000 rows read, 68.8 MB)
  validation.csv: ✅ OK (1000 rows read, 117.4 MB)
  test.csv: ✅ OK (1000 rows read, 118.0 MB)

Task A model:
  ❌ MISSING: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/model_v5_best
  Alternatives found:



In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/persian_nlp_pipline")
artifacts = base / "artifacts"

print(f"artifacts/ exists: {artifacts.exists()}\n")
if artifacts.exists():
    print("Contents:")
    for p in sorted(artifacts.iterdir()):
        if p.is_dir():
            size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
            print(f"  [dir]  {p.name}/  ({size_gb:.2f} GB)")
        else:
            print(f"  [file] {p.name}  ({p.stat().st_size/1024**2:.1f} MB)")

artifacts/ exists: True

Contents:
  [dir]  bert_rating_band/  (0.61 GB)
  [dir]  bert_recommendation/  (0.61 GB)
  [file] train_subset.parquet  (14.3 MB)


In [ ]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

rec_dir = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation")

# جستجوی هر پوشه‌ای که model.safetensors دارد
found = False
for d in rec_dir.rglob("*"):
    if d.is_file() and d.name == "model.safetensors":
        model_dir = d.parent
        print(f"Found model at: {model_dir}")
        tok = AutoTokenizer.from_pretrained(str(model_dir))
        model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
        print(f"✅ Loaded (vocab={tok.vocab_size:,}, params={sum(p.numel() for p in model.parameters()):,})")
        print(f"   id2label: {model.config.id2label}")
        found = True
        break

if not found:
    print("❌ هنوز مدل Task A پیدا نشد")

Found model at: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Loaded (vocab=5, params=162,843,651)
   id2label: {0: 'recommended', 1: 'not_recommended', 2: 'no_idea'}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation")

# بارگذاری توکنایزر از پوشه‌ی tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(model_dir / "tokenizer"))
print(f"vocab: {tokenizer.vocab_size:,}")

# بارگذاری مدل از پوشه‌ی اصلی
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
print(f"✅ Model loaded (params={sum(p.numel() for p in model.parameters()):,})")
print(f"   id2label: {model.config.id2label}")

# تست tokenization
test = tokenizer("سلام دنیا این یک تست است", return_tensors="pt")
print(f"\nTokenized: input_ids={test['input_ids'].shape}")

vocab: 100,000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded (params=162,843,651)
   id2label: {0: 'recommended', 1: 'not_recommended', 2: 'no_idea'}

Tokenized: input_ids=torch.Size([1, 8])


5

In [ ]:
# ============================================================
# 05_semi_supervised.ipynb — Cell 1: Setup
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

from pathlib import Path
import json, random
import numpy as np

DRIVE_ROOT = Path("/content/drive/MyDrive")
SPLITS_DIR = DRIVE_ROOT / "persian_nlp_pipline"
ARTIFACTS_DIR = SPLITS_DIR / "artifacts"

TRAIN_CSV              = SPLITS_DIR / "train_supervised.csv"
VAL_CSV                = SPLITS_DIR / "validation.csv"
TEST_CSV               = SPLITS_DIR / "test.csv"
PSEUDO_UNLABELED_CSV   = SPLITS_DIR / "train_pseudo_unlabeled.csv"
MANIFEST_PATH          = SPLITS_DIR / "split_manifest.json"

MODEL_TASK_A_DIR       = ARTIFACTS_DIR / "bert_recommendation"

# NEW output dir — in this Colab account's Drive, not the previous one
SEMI_DIR               = ARTIFACTS_DIR / "semi_supervised"
SEMI_DIR.mkdir(parents=True, exist_ok=True)

PSEUDO_LABELED_PARQUET = SEMI_DIR / "pseudo_labeled.parquet"
SEMI_MODEL_DIR         = SEMI_DIR / "model_semi_best"

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, PSEUDO_UNLABELED_CSV]:
    assert p.exists(), f"Missing: {p}"
assert (MODEL_TASK_A_DIR / "model.safetensors").exists(), "Task A model missing"
assert (MODEL_TASK_A_DIR / "tokenizer" / "tokenizer.json").exists(), "Task A tokenizer missing"

SPLIT_SEED = 2026
TRAIN_SEED = SPLIT_SEED
SUBSET_SEED = SPLIT_SEED

random.seed(TRAIN_SEED); np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED); torch.cuda.manual_seed_all(TRAIN_SEED)

MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

# Inference settings (fast, low-risk)
INFER_BATCH_SIZE = 128
INFER_MAX_LENGTH = 96

# Training settings (standard)
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 2e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
FP16 = True
CONF_THRESHOLD = 0.90

REC_LABELS  = ["recommended", "not_recommended", "no_idea"]
BAND_LABELS = ["low", "medium", "high"]

print(f"\nPaths OK.")
print(f"  MODEL_TASK_A_DIR      : {MODEL_TASK_A_DIR}")
print(f"  SEMI_DIR (checkpoints): {SEMI_DIR}")
print(f"\nInference: batch={INFER_BATCH_SIZE}, max_len={INFER_MAX_LENGTH}")
print(f"Training : batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}, max_len=128")
print(f"Confidence threshold: {CONF_THRESHOLD}")
print(f"Split seed: {SPLIT_SEED}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA: True
GPU: Tesla T4
GPU memory: 14.6 GB

Paths OK.
  MODEL_TASK_A_DIR      : /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation
  SEMI_DIR (checkpoints): /content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised

Inference: batch=128, max_len=96
Training : batch=16x2=32, max_len=128
Confidence threshold: 0.9
Split seed: 2026


In [ ]:
# ============================================================
# 05_semi_supervised.ipynb — Cell 2: Generate pseudo-labels
# ============================================================
# Inference on 737K unlabeled rows. Keep predictions with conf >= 0.90.
# Fallback to smaller batch if OOM.
# ============================================================

import time
import polars as pl
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 1. Load model + tokenizer -----------------------------------------
print("Loading Task A model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_TASK_A_DIR / "tokenizer"))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_TASK_A_DIR))
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"  device: {device}")
print(f"  vocab: {tokenizer.vocab_size:,}")
print(f"  id2label: {model.config.id2label}")

# --- 2. Load unlabeled data --------------------------------------------
print(f"\nLoading unlabeled data: {PSEUDO_UNLABELED_CSV.name}")
df = pl.read_csv(
    PSEUDO_UNLABELED_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    encoding="utf8-lossy",
)
print(f"  rows: {df.height:,}")

texts = df["body_normalized"].fill_null("").to_list()
ids = df["id"].to_list()

# --- 3. Inference with OOM fallback ------------------------------------
def run_inference(batch_size):
    accepted_ids, accepted_texts = [], []
    accepted_labels, accepted_confs = [], []
    n_total = len(texts)
    t0 = time.time()

    for i in range(0, n_total, batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        enc = tokenizer(
            batch_texts, truncation=True, max_length=INFER_MAX_LENGTH,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)
            max_probs, preds = probs.max(dim=-1)
        max_probs_np = max_probs.cpu().numpy()
        preds_np = preds.cpu().numpy()
        for j in range(len(batch_texts)):
            if max_probs_np[j] >= CONF_THRESHOLD:
                accepted_ids.append(batch_ids[j])
                accepted_texts.append(batch_texts[j])
                accepted_labels.append(int(preds_np[j]))
                accepted_confs.append(float(max_probs_np[j]))
        if (i // batch_size) % 200 == 0:
            elapsed = time.time() - t0
            done = min(i + batch_size, n_total)
            print(f"  {done:,}/{n_total:,} ({100*done/n_total:.1f}%)  "
                  f"accepted: {len(accepted_ids):,}  elapsed: {elapsed:.0f}s")
    return accepted_ids, accepted_texts, accepted_labels, accepted_confs

print(f"\nRunning inference (batch_size={INFER_BATCH_SIZE}, max_len={INFER_MAX_LENGTH})...")
try:
    accepted = run_inference(INFER_BATCH_SIZE)
except torch.cuda.OutOfMemoryError:
    print("\n⚠️ OOM! Reducing batch size to 64 and retrying...")
    torch.cuda.empty_cache()
    accepted = run_inference(64)

accepted_ids, accepted_texts, accepted_labels, accepted_confs = accepted

print(f"\nDone.")
print(f"  total processed: {len(texts):,}")
print(f"  accepted (conf >= {CONF_THRESHOLD}): {len(accepted_ids):,} "
      f"({100 * len(accepted_ids) / len(texts):.2f}%)")

# --- 4. Save parquet ---------------------------------------------------
id2label = model.config.id2label
pseudo_labeled = pl.DataFrame({
    "id": accepted_ids,
    "body_normalized": accepted_texts,
    "pseudo_label_id": accepted_labels,
    "confidence": accepted_confs,
}).with_columns(
    pl.col("pseudo_label_id")
      .map_elements(lambda x: id2label[int(x)], return_dtype=pl.String)
      .alias("pseudo_label")
)

print(f"\nPseudo-label distribution:")
print(pseudo_labeled.group_by("pseudo_label").agg(pl.len().alias("n")).sort("pseudo_label"))

pseudo_labeled.write_parquet(PSEUDO_LABELED_PARQUET, compression="zstd")
print(f"\nSaved to {PSEUDO_LABELED_PARQUET}")
print(f"  size: {PSEUDO_LABELED_PARQUET.stat().st_size/1024**2:.1f} MB")
print(f"  rows: {pseudo_labeled.height:,}")

Loading Task A model and tokenizer...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  device: cuda
  vocab: 100,000
  id2label: {0: 'recommended', 1: 'not_recommended', 2: 'no_idea'}

Loading unlabeled data: train_pseudo_unlabeled.csv
  rows: 737,731

Running inference (batch_size=128, max_len=96)...
  128/737,731 (0.0%)  accepted: 69  elapsed: 1s
  25,728/737,731 (3.5%)  accepted: 14,481  elapsed: 107s
  51,328/737,731 (7.0%)  accepted: 28,777  elapsed: 212s
  76,928/737,731 (10.4%)  accepted: 42,879  elapsed: 321s
  102,528/737,731 (13.9%)  accepted: 56,932  elapsed: 425s
  128,128/737,731 (17.4%)  accepted: 71,500  elapsed: 531s
  153,728/737,731 (20.8%)  accepted: 85,594  elapsed: 637s
  179,328/737,731 (24.3%)  accepted: 99,814  elapsed: 738s
  204,928/737,731 (27.8%)  accepted: 113,874  elapsed: 845s
  230,528/737,731 (31.2%)  accepted: 128,450  elapsed: 953s
  256,128/737,731 (34.7%)  accepted: 142,466  elapsed: 1061s
  281,728/737,731 (38.2%)  accepted: 156,408  elapsed: 1163s
  307,328/737,731 (41.7%)  accepted: 170,842  elapsed: 1269s
  332,928/737,731 (45.1

/tmp/ipykernel_505/2933590825.py:93: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("pseudo_label_id").map_elements(lambda x: ...)
with this one instead:
  + pl.col("pseudo_label_id").cast(pl.Int64).replace_strict(id2label)

  .map_elements(lambda x: id2label[int(x)], return_dtype=pl.String)



Pseudo-label distribution:
shape: (3, 2)
┌─────────────────┬────────┐
│ pseudo_label    ┆ n      │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ no_idea         ┆ 11631  │
│ not_recommended ┆ 32374  │
│ recommended     ┆ 363398 │
└─────────────────┴────────┘

Saved to /content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised/pseudo_labeled.parquet
  size: 12.6 MB
  rows: 407,403


In [ ]:
# ============================================================
# 05_semi_supervised.ipynb — Cell 3: Retrain with pseudo-labels
# ============================================================
# Combine supervised (500K) + pseudo-labeled (407K) and fine-tune
# the Task A model for a short additional run (5000 steps).
# Checkpoints go to SEMI_DIR (this Colab account's Drive).
# ============================================================

import time
import polars as pl
import torch
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load Task A model (continue from supervised) -------------------
print("Loading Task A model (continue from checkpoint)...")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_TASK_A_DIR / "tokenizer"))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_TASK_A_DIR))
print(f"  params: {sum(p.numel() for p in model.parameters()):,}")

# --- 2. Build combined dataset -----------------------------------------
print("\nBuilding combined dataset...")

# Load supervised train (only rows with recommendation_status)
train_sup = pl.read_csv(
    TRAIN_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    encoding="utf8-lossy",
).select(["id", "body_normalized", "recommendation_status"])
train_sup = train_sup.filter(pl.col("recommendation_status").is_not_null())
print(f"  supervised: {train_sup.height:,}")

# Load pseudo-labeled
pseudo = pl.read_parquet(PSEUDO_LABELED_PARQUET).select(
    ["id", "body_normalized", "pseudo_label"]
).rename({"pseudo_label": "recommendation_status"})
print(f"  pseudo-labeled: {pseudo.height:,}")

# Combine
combined = pl.concat([train_sup, pseudo], how="vertical")
print(f"  combined: {combined.height:,}")

# Class distribution after combining
print("\nClass distribution (combined):")
dist = combined.group_by("recommendation_status").agg(pl.len().alias("n")).sort("recommendation_status")
print(dist)
print()

# --- 3. Dataset + collator ---------------------------------------------
# Reuse ReviewDataset but only recommendation labels matter.
class RecDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length, label2id):
        self.texts = df["body_normalized"].fill_null("").to_list()
        self.labels = df["recommendation_status"].to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label2id = label2id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )
        enc["labels"] = self.label2id[self.labels[idx]]
        return enc

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_dataset = RecDataset(combined, tokenizer, 128, REC_LABEL2ID) if False else RecDataset(
    combined, tokenizer, 128, {l: i for i, l in enumerate(REC_LABELS)}
)

# Validation = same as Task A
val_df = pl.read_csv(
    VAL_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    encoding="utf8-lossy",
).select(["id", "body_normalized", "recommendation_status"])
val_df = val_df.filter(pl.col("recommendation_status").is_not_null())
val_dataset = RecDataset(val_df, tokenizer, 128, {l: i for i, l in enumerate(REC_LABELS)})
print(f"  train: {len(train_dataset):,}  val: {len(val_dataset):,}")

# --- 4. Class weights (inverse frequency on combined) -----------------
count_map = {row["recommendation_status"]: row["n"] for row in dist.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(REC_LABELS)
class_weights = torch.tensor(
    [N / (n_classes * count_map[l]) for l in REC_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")
print("\nClass weights:")
for l, w in zip(REC_LABELS, class_weights.tolist()):
    print(f"  {l:<18}  count={count_map[l]:>7,}  weight={w:.3f}")

# --- 5. WeightedTrainer ------------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 6. Optimizer with separate LR -------------------------------------
HEAD_LR = 1e-4
head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
encoder_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": head_params,    "lr": HEAD_LR,      "weight_decay": WEIGHT_DECAY},
])

# --- 7. Metrics --------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 8. TrainingArguments (5000 steps, checkpoints in SEMI_DIR) --------
training_args = TrainingArguments(
    output_dir=str(SEMI_DIR),
    max_steps=5_000,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=500,
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)
print(f"\nSchedule: max_steps=5,000, eval/save every 500 steps")

# --- 9. Trainer --------------------------------------------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# --- 10. Resume if checkpoint exists -----------------------------------
def latest_checkpoint(d):
    if not d.exists(): return None
    ckpts = [p for p in d.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")]
    if not ckpts: return None
    ckpts.sort(key=lambda p: int(p.name.split("-")[1]))
    return str(ckpts[-1])

ckpt = latest_checkpoint(SEMI_DIR)
print(f"Resume from: {ckpt or 'fresh start'}")

# --- 11. Train ---------------------------------------------------------
print("\nStarting semi-supervised training...")
t0 = time.time()
result = trainer.train(resume_from_checkpoint=ckpt)
print(f"\nFinished in {(time.time()-t0)/60:.1f} min")
print(f"  global_step: {result.global_step}")
print(f"  train_loss : {result.training_loss:.4f}")
if trainer.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer.state.best_model_checkpoint}")
if trainer.state.best_metric is not None:
    print(f"  best F1 Macro: {trainer.state.best_metric:.4f}")

# --- 12. Copy best checkpoint to SEMI_MODEL_DIR ------------------------
import shutil
best_ckpt = trainer.state.best_model_checkpoint
if best_ckpt:
    SEMI_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for f in ["model.safetensors", "config.json"]:
        src = Path(best_ckpt) / f
        if src.exists():
            shutil.copy(src, SEMI_MODEL_DIR / f)
            print(f"Copied: {f}")
    tok_src = MODEL_TASK_A_DIR / "tokenizer"
    if tok_src.exists():
        shutil.copytree(tok_src, SEMI_MODEL_DIR / "tokenizer", dirs_exist_ok=True)
        print("Copied: tokenizer/")
    print(f"Best model saved to {SEMI_MODEL_DIR}")

# --- 13. List checkpoints ---------------------------------------------
ckpts = sorted(
    [p for p in SEMI_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in SEMI_DIR ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

Loading Task A model (continue from checkpoint)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  params: 162,843,651

Building combined dataset...
  supervised: 2,950,929
  pseudo-labeled: 407,403
  combined: 3,358,332

Class distribution (combined):
shape: (3, 2)
┌───────────────────────┬─────────┐
│ recommendation_status ┆ n       │
│ ---                   ┆ ---     │
│ str                   ┆ u32     │
╞═══════════════════════╪═════════╡
│ no_idea               ┆ 344210  │
│ not_recommended       ┆ 313895  │
│ recommended           ┆ 2700227 │
└───────────────────────┴─────────┘

  train: 3,358,332  val: 780,620

Class weights:
  recommended         count=2,700,227  weight=0.415
  not_recommended     count=313,895  weight=3.566
  no_idea             count=344,210  weight=3.252

Schedule: max_steps=5,000, eval/save every 500 steps
Resume from: fresh start

Starting semi-supervised training...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
500,1.107253,0.620987,0.807771,0.682563
1000,1.212258,0.641438,0.829833,0.696552
1500,1.186370,0.615765,0.797731,0.679930
2000,1.248479,0.620711,0.824365,0.689741


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished in 57.7 min
  global_step: 2000
  train_loss : 1.1824
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised/checkpoint-1000
  best F1 Macro: 0.6966
Copied: model.safetensors
Copied: config.json
Copied: tokenizer/
Best model saved to /content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised/model_semi_best

Checkpoints in SEMI_DIR (3):
  checkpoint-1000  (1.82 GB)
  checkpoint-1500  (1.82 GB)
  checkpoint-2000  (1.82 GB)


In [ ]:
# ============================================================
# 05_semi_supervised.ipynb — Cell 4: Final evaluation on test
# ============================================================
# Evaluate supervised (Task A v5) and semi-supervised on test.csv.
# Compare with TF-IDF baseline (if metrics JSON is present).
# ============================================================

import json, time
import polars as pl
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    precision_recall_fscore_support, confusion_matrix,
)

# --- 1. Load test data -------------------------------------------------
print("Loading test.csv...")
test_df = pl.read_csv(
    TEST_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    encoding="utf8-lossy",
).select(["id", "body_normalized", "recommendation_status"])
test_df = test_df.filter(pl.col("recommendation_status").is_not_null())
print(f"  test rows: {test_df.height:,}")

texts = test_df["body_normalized"].fill_null("").to_list()
true_labels_str = test_df["recommendation_status"].to_list()

# Label mapping (same as training)
REC_LABEL2ID = {l: i for i, l in enumerate(REC_LABELS)}
ID2LABEL = {i: l for l, i in REC_LABEL2ID.items()}
true_ids = np.array([REC_LABEL2ID[l] for l in true_labels_str])

# --- 2. Evaluation helper ---------------------------------------------
def evaluate_model(model_dir, name, batch_size=64, max_length=128):
    print(f"\nEvaluating {name}...")
    tok = AutoTokenizer.from_pretrained(str(model_dir / "tokenizer"))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
    mdl.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mdl.to(device)

    all_preds = []
    t0 = time.time()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, truncation=True, max_length=max_length,
                  padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = mdl(**enc).logits
            preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())

        if (i // batch_size) % 500 == 0:
            print(f"  {min(i+batch_size, len(texts)):,}/{len(texts):,}  "
                  f"elapsed: {time.time()-t0:.0f}s")

    preds = np.array(all_preds)
    acc = accuracy_score(true_ids, preds)
    bal_acc = balanced_accuracy_score(true_ids, preds)
    f1_macro = f1_score(true_ids, preds, average="macro")
    p, r, f1, support = precision_recall_fscore_support(
        true_ids, preds, average=None, labels=list(range(len(REC_LABELS)))
    )
    per_class = {
        ID2LABEL[i]: {
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
        for i in range(len(REC_LABELS))
    }
    cm = confusion_matrix(true_ids, preds, labels=list(range(len(REC_LABELS)))).tolist()

    print(f"  ✅ {name}: acc={acc:.4f}, bal_acc={bal_acc:.4f}, macro_f1={f1_macro:.4f}")
    for label, m in per_class.items():
        print(f"     {label:<18} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")

    return {
        "model": name,
        "model_dir": str(model_dir.relative_to(DRIVE_ROOT)),
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(f1_macro),
        "per_class": per_class,
        "confusion_matrix": cm,
        "labels_order": REC_LABELS,
    }

# --- 3. Evaluate supervised and semi-supervised ------------------------
results = {}

# Supervised (Task A v5 best = checkpoint-4000, but we only kept model_v5_best or model_taskA_best)
# Check what exists
for name in ["model_v5_best", "model_taskA_best"]:
    d = MODEL_TASK_A_DIR / name
    if d.exists() and (d / "model.safetensors").exists():
        model_dir_for_sup = d
        break
else:
    model_dir_for_sup = MODEL_TASK_A_DIR  # fallback (model files at top-level)

results["supervised"] = evaluate_model(model_dir_for_sup, "supervised (Task A v5)")

# Semi-supervised
results["semi_supervised"] = evaluate_model(SEMI_MODEL_DIR, "semi-supervised")

# --- 4. Load TF-IDF baseline for comparison ----------------------------
baseline_json = ARTIFACTS_DIR / "baseline_metrics.json"
if baseline_json.exists():
    with baseline_json.open("r", encoding="utf-8") as f:
        baseline = json.load(f)
    tfidf_test = baseline["recommendation"]["test"]
    print(f"\nTF-IDF baseline (test): "
          f"acc={tfidf_test['accuracy']:.4f}, macro_f1={tfidf_test['macro_f1']:.4f}")
else:
    baseline = None
    print("\nNo baseline_metrics.json found — skipping TF-IDF comparison.")

# --- 5. Build final metrics JSON ---------------------------------------
final_metrics = {
    "test_set": {
        "n_rows": int(test_df.height),
        "labels_order": REC_LABELS,
    },
    "supervised": results["supervised"],
    "semi_supervised": results["semi_supervised"],
    "tfidf_baseline": baseline["recommendation"]["test"] if baseline else None,
    "delta_semi_vs_sup": {
        "macro_f1": results["semi_supervised"]["macro_f1"] - results["supervised"]["macro_f1"],
        "accuracy": results["semi_supervised"]["accuracy"] - results["supervised"]["accuracy"],
        "balanced_accuracy": results["semi_supervised"]["balanced_accuracy"] - results["supervised"]["balanced_accuracy"],
    },
    "delta_semi_vs_tfidf": {
        "macro_f1": results["semi_supervised"]["macro_f1"] - (baseline["recommendation"]["test"]["macro_f1"] if baseline else 0),
    } if baseline else None,
}

out_path = ARTIFACTS_DIR / "transformer_metrics.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)
print(f"\nSaved: {out_path}")
print(f"  size: {out_path.stat().st_size/1024:.1f} KB")

# --- 6. Summary table --------------------------------------------------
print("\n" + "="*70)
print("FINAL TEST RESULTS (recommendation)")
print("="*70)
print(f"{'Model':<25} {'Acc':>8} {'BalAcc':>8} {'MacroF1':>10}")
print("-"*70)
if baseline:
    bt = baseline["recommendation"]["test"]
    print(f"{'TF-IDF':<25} {bt['accuracy']:>8.4f} {bt['balanced_accuracy']:>8.4f} {bt['macro_f1']:>10.4f}")
s = results["supervised"]
print(f"{'Supervised (Task A v5)':<25} {s['accuracy']:>8.4f} {s['balanced_accuracy']:>8.4f} {s['macro_f1']:>10.4f}")
ss = results["semi_supervised"]
print(f"{'Semi-supervised':<25} {ss['accuracy']:>8.4f} {ss['balanced_accuracy']:>8.4f} {ss['macro_f1']:>10.4f}")
print("="*70)
print(f"Semi vs Supervised : Δmacro_f1 = {final_metrics['delta_semi_vs_sup']['macro_f1']:+.4f}")
if baseline:
    print(f"Semi vs TF-IDF     : Δmacro_f1 = {final_metrics['delta_semi_vs_tfidf']['macro_f1']:+.4f}")

Loading test.csv...
  test rows: 786,831

Evaluating supervised (Task A v5)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  64/786,831  elapsed: 0s
  32,064/786,831  elapsed: 113s
  64,064/786,831  elapsed: 229s
  96,064/786,831  elapsed: 347s
  128,064/786,831  elapsed: 465s
  160,064/786,831  elapsed: 583s
  192,064/786,831  elapsed: 703s
  224,064/786,831  elapsed: 823s
  256,064/786,831  elapsed: 941s
  288,064/786,831  elapsed: 1059s
  320,064/786,831  elapsed: 1179s
  352,064/786,831  elapsed: 1296s
  384,064/786,831  elapsed: 1416s
  416,064/786,831  elapsed: 1531s
  448,064/786,831  elapsed: 1650s
  480,064/786,831  elapsed: 1767s
  512,064/786,831  elapsed: 1901s
  544,064/786,831  elapsed: 2058s
  576,064/786,831  elapsed: 2149s
  608,064/786,831  elapsed: 2245s
  640,064/786,831  elapsed: 2363s
  672,064/786,831  elapsed: 2476s
  704,064/786,831  elapsed: 2605s
  736,064/786,831  elapsed: 2708s
  768,064/786,831  elapsed: 2823s
  ✅ supervised (Task A v5): acc=0.8361, bal_acc=0.7141, macro_f1=0.6941
     recommended        P=0.951 R=0.894 F1=0.922
     not_recommended    P=0.721 R=0.678 F1=0.699

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  64/786,831  elapsed: 0s
  32,064/786,831  elapsed: 120s
  64,064/786,831  elapsed: 238s
  96,064/786,831  elapsed: 358s
  128,064/786,831  elapsed: 476s
  160,064/786,831  elapsed: 594s
  192,064/786,831  elapsed: 713s
  224,064/786,831  elapsed: 834s
  256,064/786,831  elapsed: 952s
  288,064/786,831  elapsed: 1072s
  320,064/786,831  elapsed: 1197s
  352,064/786,831  elapsed: 1318s
  384,064/786,831  elapsed: 1443s
  416,064/786,831  elapsed: 1561s
  448,064/786,831  elapsed: 1684s
  480,064/786,831  elapsed: 1806s
  512,064/786,831  elapsed: 1939s
  544,064/786,831  elapsed: 2096s
  576,064/786,831  elapsed: 2186s
  608,064/786,831  elapsed: 2281s
  640,064/786,831  elapsed: 2397s
  672,064/786,831  elapsed: 2509s
  704,064/786,831  elapsed: 2638s
  736,064/786,831  elapsed: 2742s
  768,064/786,831  elapsed: 2856s
  ✅ semi-supervised: acc=0.8285, bal_acc=0.7254, macro_f1=0.6963
     recommended        P=0.957 R=0.877 F1=0.915
     not_recommended    P=0.732 R=0.686 F1=0.708
     n

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

SPLITS_DIR = Path("/content/drive/MyDrive/persian_nlp_pipline")
print(f"SPLITS_DIR exists: {SPLITS_DIR.exists()}")
print(f"test.csv: {(SPLITS_DIR / 'test.csv').exists()}")
print(f"model_semi_best: {(SPLITS_DIR / 'artifacts' / 'semi_supervised' / 'model_semi_best').exists()}")

Mounted at /content/drive
SPLITS_DIR exists: True
test.csv: True
model_semi_best: True


In [5]:
from pathlib import Path

# چک tokenizer در model_semi_best
tok1 = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised/model_semi_best/tokenizer")
print(f"model_semi_best/tokenizer: {tok1.exists()}")
if tok1.exists():
    print(f"  files: {[f.name for f in tok1.iterdir()]}")

# چک tokenizer در bert_recommendation
tok2 = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/tokenizer")
print(f"\nbert_recommendation/tokenizer: {tok2.exists()}")
if tok2.exists():
    print(f"  files: {[f.name for f in tok2.iterdir()]}")

# چک tokenizer در semi_supervised (سطح بالا)
tok3 = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised/tokenizer")
print(f"\nsemi_supervised/tokenizer: {tok3.exists()}")
if tok3.exists():
    print(f"  files: {[f.name for f in tok3.iterdir()]}")

model_semi_best/tokenizer: True
  files: ['tokenizer_config.json', 'tokenizer.json']

bert_recommendation/tokenizer: True
  files: ['tokenizer_config.json', 'tokenizer.json']

semi_supervised/tokenizer: False


In [6]:
from transformers import AutoTokenizer
from pathlib import Path

SEMI_MODEL_DIR = Path("/content/drive/MyDrive/persian_nlp_pipline/artifacts/semi_supervised/model_semi_best")
tok_dir = SEMI_MODEL_DIR / "tokenizer"

tok = AutoTokenizer.from_pretrained(str(tok_dir))
print(f"vocab: {tok.vocab_size:,}")

# تست
test = tok("سلام دنیا این یک تست است", return_tensors="pt")
print(f"input_ids shape: {test['input_ids'].shape}")

vocab: 100,000
input_ids shape: torch.Size([1, 8])


In [7]:
# ============================================================
# Error Analysis — inference (auto GPU/CPU)
# ============================================================

import time
import re
from pathlib import Path
import polars as pl
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# --- Paths ---
SPLITS_DIR = Path("/content/drive/MyDrive/persian_nlp_pipline")
TEST_CSV = SPLITS_DIR / "test.csv"
SEMI_MODEL_DIR = SPLITS_DIR / "artifacts" / "semi_supervised" / "model_semi_best"
TOKENIZER_DIR = SEMI_MODEL_DIR / "tokenizer"   # ← مسیر درست tokenizer

OUT_DIR = Path("/content/drive/MyDrive/persian_nlp_results")
OUT_DIR.mkdir(exist_ok=True)
OUT_PARQUET = OUT_DIR / "errors_analysis.parquet"

assert SEMI_MODEL_DIR.exists(), f"Missing: {SEMI_MODEL_DIR}"
assert TOKENIZER_DIR.exists(), f"Missing: {TOKENIZER_DIR}"
assert TEST_CSV.exists(), f"Missing: {TEST_CSV}"

REC_LABELS = ["recommended", "not_recommended", "no_idea"]

# --- Device selection ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    batch_size = 128
else:
    print("⚠️ No GPU — running on CPU (slower)")
    batch_size = 64
max_length = 64

# --- Load model ---
print("\nLoading model...")
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR))   # ← اصلاح شد
mdl = AutoModelForSequenceClassification.from_pretrained(str(SEMI_MODEL_DIR))
mdl.eval()
mdl.to(device)
print(f"  vocab: {tok.vocab_size:,}")   # ← باید 100,000 باشد

# --- Load test ---
print("\nLoading test.csv...")
test_df = pl.read_csv(
    TEST_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    encoding="utf8-lossy",
).select(["id", "body_normalized", "recommendation_status"])
test_df = test_df.filter(pl.col("recommendation_status").is_not_null())
print(f"  rows: {test_df.height:,}")

texts = test_df["body_normalized"].fill_null("").to_list()
true_labels = test_df["recommendation_status"].to_list()

# --- Inference ---
print(f"\nRunning inference (device={device}, batch={batch_size}, max_len={max_length})...")
all_preds, all_probs = [], []
n_batches = (len(texts) + batch_size - 1) // batch_size
t_start = time.time()

pbar = tqdm(total=n_batches, desc="Inference", unit="batch",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    enc = tok(batch, truncation=True, max_length=max_length, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = mdl(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.max(dim=-1).values.cpu().numpy().tolist())

    pbar.update(1)
    if (i // batch_size) % 50 == 0:
        done = min(i + batch_size, len(texts))
        elapsed = time.time() - t_start
        pbar.set_postfix({
            "rows": f"{done:,}",
            "elapsed": f"{elapsed/60:.1f}m",
            "rows/s": f"{done/elapsed:.0f}" if elapsed > 0 else "0",
        })

pbar.close()

id2label = mdl.config.id2label
pred_labels = [id2label[p] for p in all_preds]
total_time = time.time() - t_start
print(f"\nDone in {total_time/60:.1f} min ({total_time:.0f}s)")

# --- Build DataFrame ---
df = pl.DataFrame({
    "text": texts,
    "true": true_labels,
    "pred": pred_labels,
    "confidence": all_probs,
}).with_columns(
    (pl.col("true") != pl.col("pred")).alias("is_error"),
    pl.col("text").str.len_chars().alias("n_chars"),
    pl.col("text").str.split(by=" ").list.len().alias("n_words"),
)

n_errors = df.filter(pl.col("is_error")).height
print(f"\nTotal: {df.height:,}  Errors: {n_errors:,}  ({100*n_errors/df.height:.2f}%)")

# --- Categorize ---
def categorize(text, true_lbl, pred_lbl, n_chars):
    cats = []
    if n_chars < 15:
        cats.append("short_text")
    if re.search(r"(نمی|نبود|نیست|بد|ضعیف|افتضاح|داغون|مزخرف)", text) and pred_lbl == "recommended":
        cats.append("negation_or_negative")
    if re.search(r"(عالی|فوق‌العاده|بی‌نظیر|حرف نداره)", text) and true_lbl == "not_recommended":
        cats.append("sarcasm_or_conflict")
    if re.search(r"(ولی|اما|هرچند|با این حال)", text):
        cats.append("mixed_opinion")
    if re.search(r"(خوبه|خوب بود|بد نبود)", text) and true_lbl == "no_idea":
        cats.append("ambiguous_to_no_idea")
    if len(cats) == 0:
        cats.append("other")
    return cats

print("\nCategorizing errors...")
errors = df.filter(pl.col("is_error")).with_columns(
    pl.struct(["text", "true", "pred", "n_chars"]).map_elements(
        lambda s: categorize(s["text"], s["true"], s["pred"], s["n_chars"]),
        return_dtype=pl.List(pl.String),
    ).alias("categories")
)

cats_exploded = (
    errors.explode("categories")
    .group_by("categories")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
)
print("\nError category distribution:")
print(cats_exploded)

# --- Sample errors ---
print("\n" + "="*70)
print("SAMPLE ERRORS PER CATEGORY")
print("="*70)
for cat in cats_exploded["categories"].to_list():
    sub = errors.filter(pl.col("categories").list.contains(cat)).sort("confidence", descending=True).head(3)
    print(f"\n--- {cat} ---")
    for row in sub.iter_rows(named=True):
        print(f"  text : {row['text'][:120]!r}")
        print(f"  true : {row['true']:<18} pred: {row['pred']:<18} conf: {row['confidence']:.3f}")

# --- Save ---
errors.select(["text", "true", "pred", "confidence", "n_chars", "n_words", "categories"]).write_parquet(
    OUT_PARQUET, compression="zstd"
)
print(f"\nSaved: {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size/1024**2:.1f} MB)")


device: cuda
GPU: Tesla T4

Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  vocab: 100,000

Loading test.csv...
  rows: 786,831

Running inference (device=cuda, batch=128, max_len=64)...


Inference:   0%|          | 0/6148 [00:00<?, ?batch/s]


Done in 45.6 min (2739s)

Total: 786,831  Errors: 135,266  (17.19%)

Categorizing errors...

Error category distribution:
shape: (6, 2)
┌──────────────────────┬───────┐
│ categories           ┆ n     │
│ ---                  ┆ ---   │
│ str                  ┆ u32   │
╞══════════════════════╪═══════╡
│ other                ┆ 60732 │
│ mixed_opinion        ┆ 52376 │
│ short_text           ┆ 18562 │
│ ambiguous_to_no_idea ┆ 7696  │
│ negation_or_negative ┆ 3180  │
│ sarcasm_or_conflict  ┆ 711   │
└──────────────────────┴───────┘

SAMPLE ERRORS PER CATEGORY

--- other ---
  text : 'ترجمه افتضاح. متنش بدتر. اصلا حیف پوله. هیچ خوب نبود'
  true : no_idea            pred: not_recommended    conf: 0.994
  text : 'افتضاح. رنگش رفت. قشنگ از دور ضایعست که فیکه. شیشه شله تو سرکار موقع دست دادنی یهو بدون هیچ حرکتی افتاد شکست. کلا نخرین '
  true : recommended        pred: not_recommended    conf: 0.994
  text : 'دکمه خیلی سفته و با یکبار باز کردن دکمه میکنه و کلا جر میخوره در واقع افتضاح جنس کاور و 